# 怪盗KIDV1--多因子策略

In [1]:
# -*- coding: utf-8 -*-
"""
BigQuant 四策略簇混合策略（组内线性合成 + 组间静态资金锚 + 固定周期再平衡）

设计目标
--------
1. 七个因子只沿用用户原始代码中的因子定义/数据口径，不沿用旧策略参数。
2. 因子分成四个策略簇；簇内按手动权重做截面排名线性合成。
3. 每个策略簇独立配置市值组、组内选股比例、调仓周期和资金比例。
4. 策略簇1使用中证1000 MA60防御性+收益补偿；策略簇3使用沪深300 MA60。
5. t日收盘后形成信号并提交目标仓位，BigTrader在t+1开盘撮合。
6. 用一个BigTrader主账户执行净额持仓，同时维护四个虚拟策略账户的资金使用归因。
7. 四簇之间不做收益、夏普、回撤或市场状态驱动的动态配资；仅在固定周期日回到 TPE 搜索出的静态资金锚。
8. 单次回测审计表保留在内存；Optuna模式另外保存Trial指标与最佳配置，替代截图记录。
9. 按真实交易日预热最长504日因子，并在首个正式调仓截面强制验收覆盖率。

因子定义
--------
- hml_r_std_5m = std(high/pre_close-1, 105日) - std(low/pre_close-1, 105日)，低值优。
- exp_wgt_return_6m = 6个月成交量加权指数衰减日收益，低值优。
- bias_std_turn_5d = std(turn,5日)/std(turn,504日)-1，低值优。
- BP = 1/PB，高值优。
- Profit_G_q = 当季净利润同比增长率，高值优。
- qfa_roe = 单季度ROE（roe_avg_mrq），高值优。
- mfd_sellamt_d_sum_10 = -过去10日主力流出额合计，高值优。

重要说明
--------
- 财务因子使用BigQuant点时截面字段；请在自己的账号中运行字段检查。
- 本文件已做本地语法和静态逻辑检查，但无法在本地替代BigQuant数据权限与引擎实跑。
"""

import gc
import json
import math
import os
import time
import warnings
from copy import deepcopy
from dataclasses import dataclass
from datetime import timedelta
from statistics import NormalDist
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

print("[加载 1/4] 已进入策略脚本，正在加载科学计算组件", flush=True)

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd

print("[加载 2/4] 科学计算组件加载完成，正在检查 Optuna", flush=True)

try:
    import optuna
except Exception:
    optuna = None

print("[加载 3/4] Optuna 检查完成，正在连接 BigQuant 组件", flush=True)

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from bigquant import bigtrader, dai
except Exception as exc:
    raise ImportError("请在BigQuant AIStudio环境中运行，本策略依赖dai和BigTrader。") from exc

print("[加载 4/4] BigQuant 组件加载完成，正在初始化策略参数", flush=True)

warnings.filterwarnings("ignore")


# =============================================================================
# 1. 总体参数
# =============================================================================
# 本节集中放置所有需要人工理解或可能调整的开关。修改参数后，应重新运行完整 TPE
# 搜索；不要只把某个试验的参数手动填回最终原生回测，以免破坏“搜索—冻结—确认”顺序。

# ---- 回测范围与账户基准 ------------------------------------------------------
# START_DATE 是策略交易日程的固定锚点；各簇的“每 N 个交易日调仓”均从这里计数。
# 模拟交易任务由 BigQuant 注入 TRADING_DATE；回测时未设置该变量，使用下面的默认结束日。
# END_DATE 是因子和股票池的数据截点，绝不能超过当日，避免使用未来数据。
# 日频模拟盘的下一交易日由运行时 context.add_trading_days 获取，不能从未来行情表推断。
START_DATE = "2020-01-01"
TRADING_DATE = os.getenv("TRADING_DATE", "").strip()
END_DATE = TRADING_DATE or "2026-06-30"
# 原生 BigTrader 回测的初始账户金额（人民币）；快速模拟以 1.0 净值运行，但成本比例一致。
CAPITAL_BASE = 1_000_000
# 仅用于绩效比较，不参与股票选择、趋势判断或簇间资金分配。
BENCHMARK = "000300.SH"

# ---- 股票池、截面质量与因子处理 ----------------------------------------------
# 合格股票按总市值从小到大分成等数量的 15 组：1 为最小市值组，15 为最大市值组。
MARKET_CAP_GROUP_COUNT = 15
# 新上市不足该交易日数的股票不进入候选池，降低上市初期异常波动和数据不足影响。
MIN_LIST_DAYS = 365
# 单个选股截面至少需要的有效股票数；不足则该截面不产生新的目标持仓。
MIN_CROSS_SECTION = 100
# 单一市值组至少需要的股票数；用于避免过小分组的排名结果失真。
MIN_STOCKS_PER_GROUP = 20
# 因子去极值使用的 MAD 倍数。数值越大，保留的极端因子值越多。
WINSOR_MAD_N = 3.0

# ---- 板块范围 ---------------------------------------------------------------
# 默认所有 A 股板块均可交易。设为 True 后，该板块会从“新买入候选”中排除；
# 已持有且不可卖出的股票仍由交易约束和账本逻辑处理，不会凭空消失。
EXCLUDE_STAR_MARKET = False
EXCLUDE_CHINEXT = False
EXCLUDE_BSE = False

# ---- 成交成本与流动性限制 ----------------------------------------------------
# 买入费率、卖出费率和最低佣金均进入原生 BigTrader 回测；SELL_COST 通常包含印花税。
# 若更换券商费率或税率，应同步评估快速模拟与原生回测的差异。
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5.0
# 单笔订单可参与当日成交量的上限。较低值更保守，但可能增加未成交和资金闲置。
VOLUME_LIMIT = 0.025

# ---- 簇内仓位表达 ------------------------------------------------------------
# 无趋势防御的策略簇，正常状态下把其分配资金的 98% 投向因子股票，剩余部分保留现金。
# 该参数不是四簇间资金权重，四簇间权重由 capital_weight / TPE 静态资金锚控制。
DEFAULT_NORMAL_FACTOR_EXPOSURE = 0.98

# 趋势防御簇在风险关闭时，将因子仓位从 98% 降至 10%，并把释放的 88% 等权配置三只银行。
# 防御资产和其权重不参与 TPE 搜索；TPE 只搜索 MA 窗口与风险关闭时的因子仓位暴露。
DEFAULT_DEFENSIVE_BANKS = (
    "601398.SH",  # 工商银行
    "601328.SH",  # 交通银行
    "601988.SH",  # 中国银行
)

# ---- 运行显示与字体 ----------------------------------------------------------
# 是否在 Notebook 直接显示资金使用图；无论该开关为何，最终“四策略簇账户仓位图.png”仍会保存。
PLOT_CAPITAL_USAGE = True
# True 时额外输出每次周期再平衡和即时拒单信息；正常 TPE 搜索建议保持 False，避免日志过多。
VERBOSE_REBALANCE = False

# 中文字体：如自动探测失败，可填写BigQuant环境中字体文件的绝对路径。
CHINESE_FONT_PATH = None

# ---- 固定策略运行参数 --------------------------------------------------------
# 本文件不运行 TPE，也不重建影子账户；只执行下方写死的最优组合。
# 在模拟交易中，END_DATE 自动绑定 BigQuant 注入的 TRADING_DATE，保证只为当前交易日生成信号。
# 结果目录与运行文件同级；Notebook 中没有 __file__ 时，回退到当前工作目录。
RUN_FILE_DIRECTORY = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
OPTIMIZATION_OUTPUT_DIR = os.path.join(RUN_FILE_DIRECTORY, "固定最优参数模拟运行")
# 快速模拟的可用数据范围。起点与正式回测一致，终点严格停在 2024-12-31。
OPTIMIZATION_START_DATE = START_DATE
OPTIMIZATION_END_DATE = "2024-12-31"
# 以下三项用于通用快速指标；联合 TPE 的最终目标由后面的 JOINT_SCORE_* 单独定义。
OBJECTIVE_SHARPE_WEIGHT = 0.50
OBJECTIVE_ANNUAL_RETURN_WEIGHT = 0.50
# 用于把年化收益缩放到和夏普相近的量级，0.20 代表 20% 年化收益对应 1 个缩放单位。
OBJECTIVE_ANNUAL_RETURN_SCALE = 0.20
# 判定涨跌停开盘价的容差，避免浮点误差将涨停买入或跌停卖出误判为可成交。
FAST_LIMIT_EPS = 1e-4

# ---- 滚动样本外验证与硬约束 --------------------------------------------------
# 每折格式为：（折名、训练开始、训练结束、验证开始、验证结束）。训练区间用于保持时间顺序
# 和留痕；最终评分只汇总三个互不重叠的 2022--2024 验证区间。
# 2025 年及以后数据不参与选参，只用于最优配置冻结后的原生确认。
WALK_FORWARD_FOLDS = (
    ("第一折", "2020-01-01", "2021-12-31", "2022-01-01", "2022-12-31"),
    ("第二折", "2020-01-01", "2022-12-31", "2023-01-01", "2023-12-31"),
    ("第三折", "2020-01-01", "2023-12-31", "2024-01-01", "2024-12-31"),
)
# 每一个验证折都必须满足的最大回撤上限。0.18 = 18%，超过即视为 trial 不合格。
MAX_VALIDATION_DRAWDOWN = 0.18
# 年化总换手以“单边买卖金额合计 / 年初净值”的倍数表示；35.0 即 35 倍（3,500%），
# 与快速模拟中 gross_turnover = buy_turnover + sell_turnover 的定义一致。
MAX_VALIDATION_ANNUAL_TURNOVER = 35.0
# ---- 联合 TPE 搜索：四簇内部参数 + 静态资金锚 + 周期再平衡 --------------------
# 每个 trial 同时抽取四簇内部参数、四簇静态资金锚和簇间周期；不存在动态资金分配器、
# 收益/夏普/回撤评分配资或偏离阈值触发。
# trial 数越高，搜索更充分但运行更慢；300 是研究与运行时间之间的默认折中。
JOINT_OPTUNA_N_TRIALS = 300
# 固定随机种子使同一数据版本下的 TPE 抽样顺序可复现。
JOINT_OPTUNA_SEED = 20260723
# 【簇间参数】四簇资金回归静态锚的周期候选范围（单位：交易日，含两端）。
# 当前候选为 15、20、25、…、90；如需改变簇间再平衡周期，只修改这两行。
# 它与 OPTIMIZATION_SPACE 中各簇的 rebalance_days（簇内选股调仓周期）完全不同。
# 最优 trial 285 的簇间固定周期；只在距离上一次实际执行满 55 个交易日后再平衡。
FIXED_PERIODIC_REBALANCE_DAYS = 55
# 联合目标 = 平均夏普 + 平均年化收益 - 最差回撤惩罚 - 年化换手惩罚 - 跨折夏普离散惩罚。
# 以下为各部分权重；修改会改变最优参数的选择逻辑，应视作新的研究设定。
JOINT_SCORE_SHARPE_WEIGHT = 0.60
JOINT_SCORE_RETURN_WEIGHT = 0.40
JOINT_SCORE_DRAWDOWN_PENALTY = 0.35
JOINT_SCORE_TURNOVER_PENALTY = 0.20
JOINT_SCORE_STABILITY_PENALTY = 0.35
# 静态资金锚的边界：四簇每簇至少 10%，第一簇最多 65%；其余资金由 TPE 联合决定。
MIN_CLUSTER_CAPITAL_WEIGHT = 0.10
MAX_CLUSTER_1_CAPITAL_WEIGHT = 0.65

# ---- 各策略簇的 TPE 候选空间 -------------------------------------------------
# market_cap_groups：可选市值组组合（1 最小、15 最大）；select_pct：每个参与组内选前 k%；
# rebalance_days：该簇自己的信号/选股调仓间隔，不是簇间资金再平衡周期。
# 仅有 use_defensive_compensation=True 的簇可搜索 trend_ma_window 与
# risk_off_factor_exposure；其余参数沿用默认配置，避免无意义的维度膨胀。
# 搜索空间刻意限制为与既有有效区间相邻的候选，避免任意市值组子集爆炸。
OPTIMIZATION_SPACE = {
    "cluster_1_defensive_small": {
        # 小市值防御簇：可在市值组 1--4 中选择连续的低市值组合。
        "market_cap_groups": ((1,), (1, 2), (1, 2, 3), (1, 2, 3, 4)),
        # 每个选中市值组分别选取综合得分最高的比例，之后合并等权。
        "select_pct": (0.05, 0.075, 0.10, 0.15, 0.20),
        "rebalance_days": (15, 20, 25, 30, 40),
        # 风险开关用中证1000收盘价与其滞后均线比较；窗口越长，状态切换越平滑。
        "trend_ma_window": (40, 60, 80, 120),
        # 风险关闭时仍保留给因子股票的簇内资金比例；其余由既定防御资产承接或保留现金。
        "risk_off_factor_exposure": (0.00, 0.10, 0.20, 0.30),
    },
    "cluster_2_value_growth": {
        # 价值成长簇无趋势防御，TPE 只搜索股票池、选股比例和簇内调仓频率。
        "market_cap_groups": ((1,), (1, 2), (1, 2, 3), (1, 2, 3, 4)),
        "select_pct": (0.05, 0.075, 0.10, 0.15, 0.20),
        "rebalance_days": (20, 30, 45, 60),
    },
    "cluster_3_large_quality": {
        # 大市值质量簇只在组 13--15 的候选组合中选择，并使用沪深300趋势防御。
        "market_cap_groups": ((15,), (14, 15), (13, 14, 15)),
        "select_pct": (0.05, 0.075, 0.10, 0.15),
        "rebalance_days": (20, 30, 40, 60),
        "trend_ma_window": (40, 60, 80, 120),
        "risk_off_factor_exposure": (0.00, 0.10, 0.20, 0.30),
    },
    "cluster_4_fast_moneyflow": {
        # 快速资金流向簇保留较短的簇内调仓候选，因此通常是组合换手的主要来源之一。
        "market_cap_groups": ((1, 2), (1, 2, 3), (1, 2, 3, 4), (1, 2, 3, 4, 5)),
        "select_pct": (0.05, 0.075, 0.10, 0.15, 0.20),
        "rebalance_days": (3, 5, 10, 15),
    },
}

# 稀疏因子面板可能占用较多内存。默认只保留来源、行数和截面数摘要；
# 如确需在Notebook中检查全部因子值，可改为True。
KEEP_FACTOR_PANELS_IN_MEMORY = False

# 窗口因子的信号日筛选必须位于 QUALIFY（窗口计算之后），不能在 WHERE 或外层
# CTE 中筛选。否则 DAI 可能先只保留候选调仓日，误把 105 日和 10 日窗口算成
# “105/10 个候选调仓日”。若账户环境的 QUALIFY 窗口实现异常，可改为 True，
# 回退到逐交易日原始数据的 Python 滚动计算。
FORCE_PYTHON_ROLLING_FACTORS = False


# =============================================================================
# 2. 四策略簇参数（主要修改区域）
# =============================================================================

# ---- 固定最优四簇配置 --------------------------------------------------------
# 以下参数直接来自 2026-07-24 的联合 TPE 最优 trial 285，不再进行任何参数搜索。
# 四个 capital_weight 同时也是静态资金锚，合计严格为 100%。
#
# 字段说明：
# - factor_weights：簇内因子线性合成权重，必须恰好合计为 1；权重越高，因子对综合排名影响越大。
# - market_cap_groups：参与选股的市值组列表；每组独立排序、独立选前 select_pct，最后合并等权。
# - select_pct：每个参与市值组内选股比例，例如 0.10 即每组前 10%。
# - rebalance_days：簇内选股信号更新间隔（交易日），与 JOINT_PERIODIC_REBALANCE_* 无关。
# - capital_weight：四簇间静态资金锚的默认初始值；TPE 搜索时四项总和固定为 100%。
# - normal_factor_exposure：无趋势防御簇中，分配给因子股票的簇内资金比例。
# - use_defensive_compensation：True 时启用趋势 MA 防御；False 时不使用 trend_* / risk_off_* 字段。
# - risk_on_factor_exposure / risk_off_factor_exposure：趋势开关两种状态下的因子股票暴露。
# - risk_off_defensive_exposure：风险关闭时投向 defensive_assets 的比例；未分配部分留为现金。
CLUSTER_CONFIGS = {
    "cluster_1_defensive_small": {
        # 簇 1：最优配置为四个低市值组、每组前 5%、25 日簇内调仓、中证1000 MA40 防御。
        "display_name": "防御型小市值综合因子",
        "factor_weights": {
            "hml_r_std_5m": 0.4428565664095614,
            "exp_wgt_return_6m": 0.3658436911910037,
            "bias_std_turn_5d": 0.19129974239943487,
        },
        "market_cap_groups": [1, 2, 3, 4],
        "select_pct": 0.05,
        "rebalance_days": 25,            # 簇内每 25 个交易日更新一次目标股票。
        "capital_weight": 0.5551990309911414,
        "normal_factor_exposure": 0.98,
        "use_defensive_compensation": True,
        "trend_index": "000852.SH",    # 使用中证1000判断小市值市场趋势。
        "trend_ma_window": 40,           # 收盘价 >= MA40 为风险开启。
        "risk_on_factor_exposure": 0.98,
        "risk_off_factor_exposure": 0.10,
        "risk_off_defensive_exposure": 0.88,
        "defensive_assets": DEFAULT_DEFENSIVE_BANKS,
    },
    "cluster_2_value_growth": {
        # 簇 2：最优配置为最小市值组、每组前 20%、20 日簇内调仓；不启用趋势防御。
        "display_name": "价值成长综合因子",
        "factor_weights": {
            "BP": 0.1791659022650231,
            "Profit_G_q": 0.820834097734977,
        },
        "market_cap_groups": [1],
        "select_pct": 0.20,
        "rebalance_days": 20,
        "capital_weight": 0.15146335020995633,
        "normal_factor_exposure": 0.98,
        "use_defensive_compensation": False,
    },
    "cluster_3_large_quality": {
        # 簇 3：最优配置为市值组 13--15、每组前 7.5%、20 日簇内调仓、沪深300 MA40 防御。
        "display_name": "大市值财务质量",
        "factor_weights": {"qfa_roe": 1.0},
        "market_cap_groups": [13, 14, 15],
        "select_pct": 0.075,
        "rebalance_days": 20,
        "capital_weight": 0.14117397092985334,
        "normal_factor_exposure": 0.98,
        "use_defensive_compensation": True,
        "trend_index": "000300.SH",    # 使用沪深300判断大盘质量簇的趋势。
        "trend_ma_window": 40,
        "risk_on_factor_exposure": 0.98,
        "risk_off_factor_exposure": 0.00,
        "risk_off_defensive_exposure": 0.98,
        "defensive_assets": DEFAULT_DEFENSIVE_BANKS,
    },
    "cluster_4_fast_moneyflow": {
        # 簇 4：最优配置为市值组 1--2、每组前 15%、每 3 个交易日簇内调仓。
        "display_name": "快速资金流向",
        "factor_weights": {"mfd_sellamt_d_sum_10": 1.0},
        "market_cap_groups": [1, 2],
        "select_pct": 0.15,
        "rebalance_days": 3,
        "capital_weight": 0.15216364786904896,
        "normal_factor_exposure": 0.98,
        "use_defensive_compensation": False,
    },
}

EXPECTED_CLUSTER_FACTORS = {
    "cluster_1_defensive_small": {"hml_r_std_5m", "exp_wgt_return_6m", "bias_std_turn_5d"},
    "cluster_2_value_growth": {"BP", "Profit_G_q"},
    "cluster_3_large_quality": {"qfa_roe"},
    "cluster_4_fast_moneyflow": {"mfd_sellamt_d_sum_10"},
}


# =============================================================================
# 3. 因子构建参数
# =============================================================================

@dataclass(frozen=True)
class FactorSpec:
    name: str
    direction: int       # 1：原始值越大越优；-1：原始值越小越优
    neutralize: bool     # 是否按原因子构建口径做市值行业中性化


FACTOR_SPECS = {
    "hml_r_std_5m": FactorSpec("hml_r_std_5m", -1, True),
    "exp_wgt_return_6m": FactorSpec("exp_wgt_return_6m", -1, False),
    "bias_std_turn_5d": FactorSpec("bias_std_turn_5d", -1, True),
    "BP": FactorSpec("BP", 1, True),
    "Profit_G_q": FactorSpec("Profit_G_q", 1, False),
    "qfa_roe": FactorSpec("qfa_roe", 1, True),
    "mfd_sellamt_d_sum_10": FactorSpec("mfd_sellamt_d_sum_10", 1, True),
}

HML_LOOKBACK_DAYS = 105
HML_MIN_OBS = 80
EXP_FACTOR_MONTHS = 6
EXP_LOOKBACK_DAYS = 21 * EXP_FACTOR_MONTHS
BIAS_SHORT_WINDOW = 5
BIAS_LONG_WINDOW = 504
MONEYFLOW_WINDOW_DAYS = 10

# 因子预热必须早于正式回测。这里按真实交易日回溯，而不是简单减固定自然日。
# 额外保留80个交易日，用于吸收停牌、turn<=0和个别缺失观测。
FACTOR_WARMUP_EXTRA_TRADING_DAYS = 80
FACTOR_WARMUP_CALENDAR_BUFFER_DAYS = 1800

# 第一个正式调仓截面必须在每个候选市值组内至少保留这么多完整因子样本；
# 不满足时直接停止，避免策略静默空仓数年后才开始交易。
STRICT_FACTOR_WARMUP_VALIDATION = True
MIN_INITIAL_COMPLETE_STOCKS_PER_GROUP = MIN_STOCKS_PER_GROUP

PROFIT_FACTOR_CANDIDATES = (
    ("cn_stock_prefactors", "net_profit_yoy_mrq", "净利润同比增长率（单季度）"),
    ("cn_stock_prefactors", "net_profit_mrq_yoy", "净利润（单季度，同比增长）"),
    ("cn_stock_prefactors", "net_profit_to_parent_yoy_mrq", "归母净利润同比增长率（单季度）"),
    ("cn_stock_prefactors", "net_profit_to_parent_mrq_yoy", "归母净利润（单季度，同比增长）"),
    ("cn_stock_prefactors", "net_profit_yoy_lf", "净利润同比增长率（最新一期）"),
    ("cn_stock_prefactors", "net_profit_lf_yoy", "净利润（最新一期，同比增长）"),
    ("cn_stock_prefactors", "net_profit_to_parent_yoy_lf", "归母净利润同比增长率（最新一期）"),
    ("cn_stock_prefactors", "net_profit_to_parent_lf_yoy", "归母净利润（最新一期，同比增长）"),
    ("cn_stock_factors_financial_indicators", "net_profit_yoy_mrq", "净利润同比增长率（单季度）"),
    ("cn_stock_factors_financial_indicators", "net_profit_mrq_yoy", "净利润（单季度，同比增长）"),
    ("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy_mrq", "归母净利润同比增长率（单季度）"),
    ("cn_stock_factors_financial_indicators", "net_profit_to_parent_mrq_yoy", "归母净利润（单季度，同比增长）"),
    ("cn_stock_factors_financial_indicators", "net_profit_yoy_lf", "净利润同比增长率（最新一期）"),
    ("cn_stock_factors_financial_indicators", "net_profit_lf_yoy", "净利润（最新一期，同比增长）"),
    ("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy_lf", "归母净利润同比增长率（最新一期）"),
    ("cn_stock_factors_financial_indicators", "net_profit_to_parent_lf_yoy", "归母净利润（最新一期，同比增长）"),
    ("cn_stock_prefactors", "net_profit_yoy", "净利润同比增长率"),
    ("cn_stock_prefactors", "net_profit_to_parent_yoy", "归母净利润同比增长率"),
    ("cn_stock_factors_financial_indicators", "net_profit_yoy", "净利润同比增长率"),
    ("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy", "归母净利润同比增长率"),
)


# 运行产物全部保留在内存中。
RESEARCH_ARTIFACTS: Dict[str, object] = {}
_RUNTIME_DATA: Dict[str, object] = {}
_ACTIVE_CONFIGS: Dict[str, dict] = {}
_T0 = time.time()


# =============================================================================
# 4. 参数校验与通用函数
# =============================================================================

def progress(message: str) -> None:
    elapsed = int(time.time() - _T0)
    print(f"[{elapsed // 60:02d}:{elapsed % 60:02d}] {message}", flush=True)


def date_text(value) -> str:
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def date_sql(values: Iterable) -> str:
    unique = sorted({date_text(value) for value in values})
    if not unique:
        raise ValueError("日期列表不能为空。")
    return ", ".join(f"'{value}'" for value in unique)


def query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    result = dai.query(sql, filters=filters) if filters is not None else dai.query(sql)
    return result.df()


def result_output_directory() -> str:
    """创建与运行文件同级的中文结果目录。"""
    path = os.path.abspath(os.path.expanduser(OPTIMIZATION_OUTPUT_DIR))
    os.makedirs(path, exist_ok=True)
    return path


def save_result_table(frame: pd.DataFrame, filename: str) -> str:
    """以 UTF-8-SIG 保存可直接被 Excel 打开的研究表。"""
    path = os.path.join(result_output_directory(), filename)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def save_result_json(payload: Mapping, filename: str) -> str:
    path = os.path.join(result_output_directory(), filename)
    with open(path, "w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2, default=str)
    return path


def first_success_query(candidates: Sequence[Tuple[str, Optional[dict], str]]) -> Tuple[pd.DataFrame, str]:
    errors = []
    for sql, filters, label in candidates:
        try:
            frame = query_df(sql, filters=filters)
            if frame is not None and not frame.empty:
                return frame, label
            errors.append(f"{label}: 查询成功但为空")
        except Exception as exc:
            errors.append(f"{label}: {str(exc).splitlines()[-1][:240]}")
    raise RuntimeError("所有候选查询均失败：\n" + "\n".join(errors))


def validate_configs(configs: Mapping[str, dict]) -> Dict[str, dict]:
    configs = deepcopy(dict(configs))
    if set(configs) != set(CLUSTER_CONFIGS):
        missing = sorted(set(CLUSTER_CONFIGS) - set(configs))
        extra = sorted(set(configs) - set(CLUSTER_CONFIGS))
        raise ValueError(f"策略簇必须保持四组完整。缺失={missing}，多余={extra}")

    total_capital_weight = 0.0
    for cluster_name, config in configs.items():
        weights = {str(k): float(v) for k, v in config["factor_weights"].items()}
        unknown = sorted(set(weights) - set(FACTOR_SPECS))
        if unknown:
            raise ValueError(f"{cluster_name}包含未知因子：{unknown}")
        if set(weights) != EXPECTED_CLUSTER_FACTORS[cluster_name]:
            raise ValueError(
                f"{cluster_name}因子成员必须为{sorted(EXPECTED_CLUSTER_FACTORS[cluster_name])}，"
                f"当前为{sorted(weights)}。"
            )
        if not weights or any((not np.isfinite(v) or v < 0) for v in weights.values()):
            raise ValueError(f"{cluster_name}因子权重必须为非负有限数。")
        weight_sum = sum(weights.values())
        if not np.isclose(weight_sum, 1.0, atol=1e-8):
            raise ValueError(f"{cluster_name}因子权重之和必须为1，当前为{weight_sum:.8f}。")
        config["factor_weights"] = weights

        groups = sorted({int(x) for x in config["market_cap_groups"]})
        invalid = [x for x in groups if x < 1 or x > MARKET_CAP_GROUP_COUNT]
        if not groups or invalid:
            raise ValueError(f"{cluster_name}市值组非法：{invalid or groups}")
        config["market_cap_groups"] = groups

        select_pct = float(config["select_pct"])
        if not 0 < select_pct <= 1:
            raise ValueError(f"{cluster_name}.select_pct必须位于(0,1]。")
        if int(config["rebalance_days"]) < 1:
            raise ValueError(f"{cluster_name}.rebalance_days必须为正整数。")
        capital_weight = float(config["capital_weight"])
        if not 0 <= capital_weight <= 1:
            raise ValueError(f"{cluster_name}.capital_weight必须位于[0,1]。")
        total_capital_weight += capital_weight

        if bool(config.get("use_defensive_compensation", False)):
            required = (
                "trend_index", "trend_ma_window", "risk_on_factor_exposure",
                "risk_off_factor_exposure", "risk_off_defensive_exposure", "defensive_assets",
            )
            missing = [key for key in required if key not in config]
            if missing:
                raise ValueError(f"{cluster_name}防御参数缺失：{missing}")
            on = float(config["risk_on_factor_exposure"])
            off = float(config["risk_off_factor_exposure"])
            defense = float(config["risk_off_defensive_exposure"])
            if not (0 <= off < on <= 1):
                raise ValueError(f"{cluster_name}风险开关仓位不合法。")
            if defense < 0 or off + defense > 1 + 1e-10:
                raise ValueError(f"{cluster_name}防御仓位不合法。")
            if not config["defensive_assets"]:
                raise ValueError(f"{cluster_name}防御资产不能为空。")
        else:
            exposure = float(config.get("normal_factor_exposure", DEFAULT_NORMAL_FACTOR_EXPOSURE))
            if not 0 <= exposure <= 1:
                raise ValueError(f"{cluster_name}.normal_factor_exposure必须位于[0,1]。")

    if not np.isclose(total_capital_weight, 1.0, atol=1e-10):
        raise ValueError(f"四个策略簇静态资金锚权重之和必须为100%，当前为{total_capital_weight:.2%}。")
    return configs


def board_allowed(instrument: str) -> bool:
    code = str(instrument).upper()
    number = code.split(".")[0]
    if EXCLUDE_BSE and (code.endswith(".BJ") or number.startswith(("43", "83", "87", "88", "92"))):
        return False
    if EXCLUDE_STAR_MARKET and code.endswith(".SH") and number.startswith(("688", "689")):
        return False
    if EXCLUDE_CHINEXT and code.endswith(".SZ") and number.startswith(("300", "301")):
        return False
    return True


def robust_zscore(values: pd.Series, mad_n: float = WINSOR_MAD_N) -> pd.Series:
    x = pd.to_numeric(values, errors="coerce").astype(float)
    out = pd.Series(np.nan, index=x.index, dtype=float)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out
    sample = x.loc[valid]
    median = sample.median()
    mad = (sample - median).abs().median()
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        low, high = median - mad_n * scale, median + mad_n * scale
    else:
        low, high = sample.quantile([0.01, 0.99]).tolist()
    clipped = sample.clip(low, high)
    std = clipped.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out.loc[valid] = (clipped - clipped.mean()) / std
    return out


def neutralize_score(factor_z: pd.Series, market_cap: pd.Series, industry: pd.Series) -> pd.Series:
    y = pd.to_numeric(factor_z, errors="coerce").astype(float)
    log_cap = np.log(pd.to_numeric(market_cap, errors="coerce").astype(float).clip(lower=1.0))
    industry_values = industry.fillna("未知").astype(str)
    dummies = pd.get_dummies(industry_values, prefix="industry", drop_first=True, dtype=float)
    design = pd.concat(
        [pd.Series(1.0, index=y.index, name="intercept"), log_cap.rename("log_cap"), dummies],
        axis=1,
    )
    valid = np.isfinite(y) & np.isfinite(design).all(axis=1)
    residual = pd.Series(np.nan, index=y.index, dtype=float)
    if valid.sum() <= design.shape[1] + 5:
        return residual
    x_array = design.loc[valid].to_numpy(dtype=float)
    y_array = y.loc[valid].to_numpy(dtype=float)
    beta = np.linalg.lstsq(x_array, y_array, rcond=None)[0]
    residual.loc[valid] = y_array - x_array @ beta
    return robust_zscore(residual)


def assign_market_cap_groups(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    rank = frame["total_market_cap"].rank(method="first", ascending=True)
    if len(frame) < MARKET_CAP_GROUP_COUNT * MIN_STOCKS_PER_GROUP:
        return pd.DataFrame()
    try:
        frame["market_cap_group"] = pd.qcut(
            rank,
            q=MARKET_CAP_GROUP_COUNT,
            labels=list(range(1, MARKET_CAP_GROUP_COUNT + 1)),
        ).astype(int)
    except ValueError:
        return pd.DataFrame()
    return frame


def current_positions(context) -> Dict[str, object]:
    try:
        positions = context.portfolio.positions
        return dict(positions) if positions is not None else {}
    except Exception:
        pass
    for method_name in ("get_positions", "get_account_positions"):
        if hasattr(context, method_name):
            try:
                result = getattr(context, method_name)()
                return dict(result) if result is not None else {}
            except Exception:
                pass
    return {}


def position_amount(position) -> float:
    for attr in ("amount", "current_qty", "volume", "quantity"):
        if hasattr(position, attr):
            try:
                return float(getattr(position, attr) or 0.0)
            except Exception:
                pass
    return 0.0


def position_market_value(position) -> float:
    for attr in ("market_value", "position_value", "value"):
        if hasattr(position, attr):
            try:
                return float(getattr(position, attr) or 0.0)
            except Exception:
                pass
    return 0.0


def portfolio_value(context) -> float:
    try:
        return float(context.portfolio.portfolio_value)
    except Exception:
        pass
    try:
        return float(context.get_balance().total_asset)
    except Exception:
        return float(CAPITAL_BASE)


# =============================================================================
# 5. 交易日程
# =============================================================================

def query_trading_calendar() -> List[pd.Timestamp]:
    """查询交易日历。

    日频模拟盘仅把 END_DATE 及以前的数据用于选股。下一交易日开盘日期只能在
    运行时通过 context.add_trading_days 获取，不能读取未来行情或未来因子。
    """
    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_bar1d
    WHERE date >= '{START_DATE}' AND date <= '{END_DATE}'
    ORDER BY date
    """
    frame = query_df(sql, filters={"date": [START_DATE, END_DATE]})
    if frame.empty:
        raise ValueError("回测区间交易日历为空。")
    return sorted(pd.to_datetime(frame["date"]).dt.normalize().drop_duplicates().tolist())


def query_factor_warmup_start() -> str:
    """按真实交易日确定最长因子窗口的预热起点。"""
    required = max(
        HML_LOOKBACK_DAYS,
        EXP_LOOKBACK_DAYS + 1,
        BIAS_LONG_WINDOW,
        MONEYFLOW_WINDOW_DAYS,
    ) + int(FACTOR_WARMUP_EXTRA_TRADING_DAYS)
    provisional_start = date_text(
        pd.Timestamp(START_DATE) - timedelta(days=FACTOR_WARMUP_CALENDAR_BUFFER_DAYS)
    )
    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_bar1d
    WHERE date >= '{provisional_start}' AND date < '{START_DATE}'
    ORDER BY date
    """
    frame = query_df(sql, filters={"date": [provisional_start, START_DATE]})
    dates = sorted(
        pd.to_datetime(frame.get("date", pd.Series(dtype="datetime64[ns]")))
        .dt.normalize()
        .dropna()
        .drop_duplicates()
        .tolist()
    )
    if len(dates) < required:
        raise ValueError(
            "因子预热交易日不足："
            f"需要至少{required}个回测开始日前交易日，实际只有{len(dates)}个。"
            "请扩大FACTOR_WARMUP_CALENDAR_BUFFER_DAYS或检查历史行情权限。"
        )
    warmup_start = date_text(dates[-required])
    progress(
        f"因子预热：{warmup_start}至{START_DATE}，"
        f"共使用{required}个回测开始日前交易日（最长窗口{BIAS_LONG_WINDOW}日）"
    )
    return warmup_start


def build_schedules(trade_dates: Sequence[pd.Timestamp], configs: Mapping[str, dict]):
    schedules = {}
    next_date_map = {
        date_text(trade_dates[index]): date_text(trade_dates[index + 1])
        for index in range(len(trade_dates) - 1)
    }
    for cluster_name, config in configs.items():
        n = int(config["rebalance_days"])
        signals = trade_dates[::n]
        # 回测末日没有下一根日线，不能生成无法验证成交的订单；日频模拟盘例外，
        # 它在运行时通过 context.add_trading_days 把当日收盘信号映射到下一开盘。
        signals = [
            value for value in signals
            if date_text(value) <= END_DATE
            and (
                date_text(value) in next_date_map
                or (bool(TRADING_DATE) and date_text(value) == END_DATE)
            )
        ]
        schedules[cluster_name] = {
            "signal_dates": [date_text(value) for value in signals],
            "expected_trade_dates": {date_text(value): next_date_map[date_text(value)] for value in signals},
        }
    return schedules, next_date_map


# =============================================================================
# 6. 点时股票池与七个原始因子
# =============================================================================

def query_base_panel(signal_dates: Sequence[str]) -> pd.DataFrame:
    dates = date_sql(signal_dates)
    candidates = []
    for industry_field in ("sw2021_level1", "sw2014_level1", "cs_level1_name"):
        sql = f"""
        SELECT
            date,
            instrument,
            total_market_cap,
            float_market_cap,
            {industry_field} AS industry,
            st_status,
            suspended,
            list_sector,
            list_days,
            pb,
            roe_avg_mrq
        FROM cn_stock_prefactors
        WHERE date IN ({dates})
        ORDER BY date, instrument
        """
        candidates.append((sql, {"date": [min(signal_dates), max(signal_dates)]}, industry_field))
    frame, source = first_success_query(candidates)
    progress(f"股票池基础字段行业来源：{source}")
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    for col in ("total_market_cap", "float_market_cap", "st_status", "suspended", "list_days", "pb", "roe_avg_mrq"):
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    frame["industry"] = frame["industry"].fillna("未知").astype(str)
    frame = frame.drop_duplicates(["date", "instrument"], keep="last")
    return frame


def query_hml_factor(signal_dates: Sequence[str], query_start: str) -> pd.DataFrame:
    """由high、low、pre_close原始行情构建HML波动率差。"""
    dates = date_sql(signal_dates)
    # hml只需要105个交易日，单独使用较短预热区间，避免Python兜底读取900天全市场数据。
    hml_query_start = date_text(pd.Timestamp(min(signal_dates)) - timedelta(days=300))
    candidates = []
    sql_pre_close = f"""
        SELECT
            date,
            instrument,
            stddev_samp(high / NULLIF(pre_close, 0) - 1) OVER factor_window
            - stddev_samp(low / NULLIF(pre_close, 0) - 1) OVER factor_window AS hml_r_std_5m
        FROM cn_stock_bar1d
        WHERE date >= '{hml_query_start}'
          AND date <= '{END_DATE}'
          AND high > 0 AND low > 0 AND close > 0 AND pre_close > 0
        WINDOW factor_window AS (
            PARTITION BY instrument ORDER BY date
            ROWS BETWEEN {HML_LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
        )
        QUALIFY date IN ({dates})
          AND count(high / NULLIF(pre_close, 0) - 1) OVER factor_window >= {HML_MIN_OBS}
        ORDER BY date, instrument
    """
    sql_lag_close = f"""
        WITH base AS (
            SELECT
                date,
                instrument,
                high,
                low,
                close,
                lag(close) OVER (PARTITION BY instrument ORDER BY date) AS pre_close
            FROM cn_stock_bar1d
            WHERE date >= '{hml_query_start}'
              AND date <= '{END_DATE}'
              AND high > 0
              AND low > 0
              AND close > 0
        ), daily AS (
            SELECT
                date,
                instrument,
                high / NULLIF(pre_close, 0) - 1 AS high_r,
                low / NULLIF(pre_close, 0) - 1 AS low_r
            FROM base
            WHERE pre_close > 0
        )
        SELECT
            date,
            instrument,
            stddev_samp(high_r) OVER factor_window
            - stddev_samp(low_r) OVER factor_window AS hml_r_std_5m
        FROM daily
        WINDOW factor_window AS (
            PARTITION BY instrument ORDER BY date
            ROWS BETWEEN {HML_LOOKBACK_DAYS - 1} PRECEDING AND CURRENT ROW
        )
        QUALIFY date IN ({dates})
          AND count(high_r) OVER factor_window >= {HML_MIN_OBS}
        ORDER BY date, instrument
    """
    candidates.extend(
        [
            (sql_pre_close, None, "SQL显式历史区间/pre_close"),
            (sql_lag_close, None, "SQL显式历史区间/lag(close)"),
        ]
    )
    try:
        if FORCE_PYTHON_ROLLING_FACTORS:
            raise RuntimeError(
                "已启用逐交易日 Python 滚动，避免 DAI 对 date IN 的窗口谓词下推"
            )
        frame, source = first_success_query(candidates)
    except RuntimeError as sql_error:
        progress(
            "hml_r_std_5m的DAI复杂窗口没有返回调仓截面（并非查询同名成品因子），"
            f"改用原始high/low/pre_close在Python中计算：{sql_error}"
        )
        bar_candidates = [
            (
                f"""
                SELECT date, instrument, high, low, close, pre_close
                FROM cn_stock_bar1d
                WHERE date >= '{hml_query_start}' AND date <= '{END_DATE}'
                ORDER BY instrument, date
                """,
                None,
                "Python兜底/pre_close",
            ),
            (
                f"""
                SELECT date, instrument, high, low, close
                FROM cn_stock_bar1d
                WHERE date >= '{hml_query_start}' AND date <= '{END_DATE}'
                ORDER BY instrument, date
                """,
                None,
                "Python兜底/shift(close)",
            ),
        ]
        bar, source = first_success_query(bar_candidates)
        bar["date"] = pd.to_datetime(bar["date"]).dt.normalize()
        bar["instrument"] = bar["instrument"].astype(str)
        for column in ("high", "low", "close", "pre_close"):
            if column in bar.columns:
                bar[column] = pd.to_numeric(bar[column], errors="coerce", downcast="float")
        bar = bar.dropna(subset=["date", "instrument", "high", "low", "close"])
        bar = bar[(bar["high"] > 0) & (bar["low"] > 0) & (bar["close"] > 0)]
        bar = bar.sort_values(["instrument", "date"], kind="mergesort").reset_index(drop=True)
        shifted_close = bar.groupby("instrument", sort=False)["close"].shift(1)
        if "pre_close" not in bar.columns:
            bar["pre_close"] = shifted_close
        else:
            # 某些数据版本只有部分行缺失pre_close；逐行回填，不能只判断整列是否为空。
            bar["pre_close"] = bar["pre_close"].where(bar["pre_close"] > 0, shifted_close)
        bar["high_r"] = bar["high"] / bar["pre_close"] - 1.0
        bar["low_r"] = bar["low"] / bar["pre_close"] - 1.0
        bar.loc[~np.isfinite(bar["high_r"]), "high_r"] = np.nan
        bar.loc[~np.isfinite(bar["low_r"]), "low_r"] = np.nan
        bar["high_r_std"] = (
            bar.groupby("instrument", sort=False)["high_r"]
            .rolling(HML_LOOKBACK_DAYS, min_periods=HML_MIN_OBS)
            .std()
            .reset_index(level=0, drop=True)
        )
        bar["low_r_std"] = (
            bar.groupby("instrument", sort=False)["low_r"]
            .rolling(HML_LOOKBACK_DAYS, min_periods=HML_MIN_OBS)
            .std()
            .reset_index(level=0, drop=True)
        )
        bar["hml_r_std_5m"] = bar["high_r_std"] - bar["low_r_std"]
        signal_set = set(pd.to_datetime(signal_dates))
        frame = bar.loc[
            bar["date"].isin(signal_set),
            ["date", "instrument", "hml_r_std_5m"],
        ].copy()
        del bar
        gc.collect()
    progress(f"hml_r_std_5m构建来源：{source}")
    return clean_factor_frame(frame, "hml_r_std_5m")


def query_exp_factor(signal_dates: Sequence[str], query_start: str) -> pd.DataFrame:
    """由close、turn及其历史滞后项构建6个月成交量加权衰减收益。"""
    dates = date_sql(signal_dates)
    exp_query_start = date_text(pd.Timestamp(min(signal_dates)) - timedelta(days=360))
    numerator = []
    denominator = []
    for i in range(EXP_LOOKBACK_DAYS):
        decay = math.exp(-i / EXP_FACTOR_MONTHS / 4.0)
        close_i = "close" if i == 0 else f"m_lag(close,{i})"
        close_prev = f"m_lag(close,{i + 1})"
        turn_i = "turn" if i == 0 else f"m_lag(turn,{i})"
        ret_i = f"(({close_i}/NULLIF({close_prev},0))-1.0)"
        numerator.append(f"COALESCE({decay:.12g}*{turn_i}*{ret_i},0.0)")
        denominator.append(f"COALESCE({decay:.12g}*{turn_i},0.0)")
    num_expr = "+".join(numerator)
    den_expr = "+".join(denominator)
    sql = f"""
    WITH daily_factor AS (
        SELECT
            date,
            instrument,
            ({num_expr}) / NULLIF(({den_expr}),0) AS exp_wgt_return_6m
        FROM cn_stock_prefactors
        WHERE date >= '{exp_query_start}' AND date <= '{END_DATE}'
    )
    SELECT date, instrument, exp_wgt_return_6m
    FROM daily_factor
    WHERE date IN ({dates}) AND exp_wgt_return_6m IS NOT NULL
    ORDER BY date, instrument
    """
    frame = query_df(sql)
    progress("exp_wgt_return_6m构建来源：cn_stock_prefactors.close/turn + m_lag")
    return clean_factor_frame(frame, "exp_wgt_return_6m")


def query_bias_factor(signal_dates: Sequence[str], query_start: str) -> pd.DataFrame:
    """由turn原始字段构建std(turn,5)/std(turn,504)-1，不读取同名成品因子。"""
    dates = date_sql(signal_dates)
    # 保持用户原始bias代码的关键结构：窗口计算层不写日期WHERE；独立日期表JOIN截面；
    # 连续历史范围通过DAI filters提供，避免m_窗口算子被稀疏调仓日期谓词下推。
    sql = f"""
    WITH selected_dates AS (
        SELECT DISTINCT date
        FROM cn_stock_bar1d
        WHERE date IN ({dates})
    ),
    daily_factor AS (
        SELECT
            date,
            instrument,
            m_stddev(if(turn > 0, turn, NULL), {BIAS_SHORT_WINDOW}) AS std_short,
            m_stddev(if(turn > 0, turn, NULL), {BIAS_LONG_WINDOW}) AS std_long
        FROM cn_stock_prefactors
    )
    SELECT
        f.date,
        f.instrument,
        f.std_short / NULLIF(f.std_long, 0) - 1 AS bias_std_turn_5d
    FROM daily_factor f
    INNER JOIN selected_dates d
        ON f.date = d.date
    ORDER BY f.date, f.instrument
    """
    try:
        frame = query_df(sql, filters={"date": [query_start, END_DATE]})
        factor_values = (
            pd.Series(dtype=float)
            if frame is None or frame.empty
            else pd.to_numeric(frame["bias_std_turn_5d"], errors="coerce")
        )
        first_signal = pd.Timestamp(min(signal_dates)).normalize()
        first_dates = (
            pd.Series(dtype="datetime64[ns]")
            if frame is None or frame.empty
            else pd.to_datetime(frame["date"]).dt.normalize()
        )
        first_valid_count = int(
            factor_values[first_dates.eq(first_signal)].replace([np.inf, -np.inf], np.nan).notna().sum()
        )
        if first_valid_count < MIN_CROSS_SECTION:
            raise ValueError(
                f"首个信号日{date_text(first_signal)}只有{first_valid_count}个有效值，"
                f"低于最低要求{MIN_CROSS_SECTION}；DAI窗口预热未通过验收"
            )
        source = "cn_stock_prefactors.turn / DAI m_stddev(5,504)"
    except Exception as sql_error:
        progress(
            "bias_std_turn_5d的DAI窗口方案未返回有效值，"
            f"改用原始turn字段在Python中滚动计算：{sql_error}"
        )
        raw_candidates = [
            (
                f"""
                SELECT date, instrument, turn
                FROM cn_stock_prefactors
                WHERE date >= '{query_start}' AND date <= '{END_DATE}'
                ORDER BY instrument, date
                """,
                {"date": [query_start, END_DATE]},
                "Python兜底/显式日期区间",
            ),
            (
                """
                SELECT date, instrument, turn
                FROM cn_stock_prefactors
                ORDER BY instrument, date
                """,
                {"date": [query_start, END_DATE]},
                "Python兜底/DAI filters日期区间",
            ),
        ]
        raw, fallback_source = first_success_query(raw_candidates)
        raw["date"] = pd.to_datetime(raw["date"]).dt.normalize()
        raw["instrument"] = raw["instrument"].astype(str)
        raw["turn"] = pd.to_numeric(raw["turn"], errors="coerce", downcast="float")
        raw = raw.dropna(subset=["date", "instrument"]).sort_values(
            ["instrument", "date"], kind="mergesort"
        ).reset_index(drop=True)
        # 与原始SQL的if(turn > 0, turn, NULL)完全一致。
        raw["turn_valid"] = raw["turn"].where(raw["turn"] > 0)
        grouped_turn = raw.groupby("instrument", sort=False)["turn_valid"]
        # rolling窗口长度按交易行数控制；窗口内turn<=0继续按原始口径作为NULL剔除。
        # 先允许std忽略窗口内的少量NULL，再强制要求已经积累完整的5/504个历史交易行。
        raw["history_row_count"] = raw.groupby("instrument", sort=False).cumcount() + 1
        raw["std_short"] = (
            grouped_turn.rolling(BIAS_SHORT_WINDOW, min_periods=2)
            .std(ddof=1)
            .reset_index(level=0, drop=True)
        )
        raw["std_long"] = (
            grouped_turn.rolling(BIAS_LONG_WINDOW, min_periods=2)
            .std(ddof=1)
            .reset_index(level=0, drop=True)
        )
        raw.loc[raw["history_row_count"] < BIAS_SHORT_WINDOW, "std_short"] = np.nan
        raw.loc[raw["history_row_count"] < BIAS_LONG_WINDOW, "std_long"] = np.nan
        raw["bias_std_turn_5d"] = raw["std_short"] / raw["std_long"] - 1.0
        raw.loc[~np.isfinite(raw["bias_std_turn_5d"]), "bias_std_turn_5d"] = np.nan
        signal_set = set(pd.to_datetime(signal_dates))
        frame = raw.loc[
            raw["date"].isin(signal_set),
            ["date", "instrument", "bias_std_turn_5d"],
        ].copy()
        del raw
        gc.collect()
        source = f"cn_stock_prefactors.turn / {fallback_source}"
        first_signal = pd.Timestamp(min(signal_dates)).normalize()
        first_valid_count = int(
            pd.to_numeric(
                frame.loc[frame["date"].eq(first_signal), "bias_std_turn_5d"],
                errors="coerce",
            ).replace([np.inf, -np.inf], np.nan).notna().sum()
        )
        if first_valid_count < MIN_CROSS_SECTION:
            raise ValueError(
                f"bias_std_turn_5d完成连续日频预热后，首个信号日{date_text(first_signal)}"
                f"仍只有{first_valid_count}个有效值，低于最低要求{MIN_CROSS_SECTION}。"
                "请检查cn_stock_prefactors.turn的历史覆盖和字段权限。"
            )
    progress(f"bias_std_turn_5d构建来源：{source}")
    return clean_factor_frame(frame, "bias_std_turn_5d")


def query_bp_factor(signal_dates: Sequence[str]) -> Tuple[pd.DataFrame, str]:
    """按照原始BP代码优先读取估值表PB，再在Python中构建BP=1/PB。"""
    dates = date_sql(signal_dates)
    candidates = [
        (
            f"""
            SELECT date, instrument, pb
            FROM cn_stock_valuation
            WHERE date IN ({dates}) AND pb IS NOT NULL
            ORDER BY date, instrument
            """,
            {"date": [min(signal_dates), max(signal_dates)]},
            "cn_stock_valuation.pb",
        ),
        # 仅作为不同账号数据权限下的兼容回退；计算公式仍然是1/PB。
        (
            f"""
            SELECT date, instrument, pb
            FROM cn_stock_prefactors
            WHERE date IN ({dates}) AND pb IS NOT NULL
            ORDER BY date, instrument
            """,
            {"date": [min(signal_dates), max(signal_dates)]},
            "cn_stock_prefactors.pb（兼容回退）",
        ),
    ]
    frame, source = first_success_query(candidates)
    frame["pb"] = pd.to_numeric(frame["pb"], errors="coerce")
    frame["BP"] = np.where(frame["pb"] > 0, 1.0 / frame["pb"], np.nan)
    progress(f"BP构建来源：{source}；公式=1/PB")
    return clean_factor_frame(frame[["date", "instrument", "BP"]], "BP"), source


def query_profit_factor(signal_dates: Sequence[str]) -> Tuple[pd.DataFrame, str]:
    dates = date_sql(signal_dates)
    candidates = []
    for table, field, description in PROFIT_FACTOR_CANDIDATES:
        sql = f"""
        SELECT date, instrument, {field} AS Profit_G_q
        FROM {table}
        WHERE date IN ({dates}) AND {field} IS NOT NULL
        ORDER BY date, instrument
        """
        candidates.append((sql, {"date": [min(signal_dates), max(signal_dates)]}, f"{table}.{field}（{description}）"))
    frame, source = first_success_query(candidates)
    return clean_factor_frame(frame, "Profit_G_q"), source


def query_moneyflow_factor(signal_dates: Sequence[str], query_start: str) -> pd.DataFrame:
    """由主力流出额原始字段构建负的10日累计值。"""
    dates = date_sql(signal_dates)
    moneyflow_query_start = date_text(
        pd.Timestamp(min(signal_dates)) - timedelta(days=max(60, MONEYFLOW_WINDOW_DAYS * 5))
    )
    sql = f"""
    SELECT
        date,
        instrument,
        -SUM(outflow_amount_main) OVER factor_window AS mfd_sellamt_d_sum_10
    FROM cn_stock_moneyflow
    WHERE date >= '{moneyflow_query_start}' AND date <= '{END_DATE}'
    WINDOW factor_window AS (
        PARTITION BY instrument ORDER BY date
        ROWS BETWEEN {MONEYFLOW_WINDOW_DAYS - 1} PRECEDING AND CURRENT ROW
    )
    QUALIFY date IN ({dates})
      AND COUNT(outflow_amount_main) OVER factor_window = {MONEYFLOW_WINDOW_DAYS}
    ORDER BY date, instrument
    """
    try:
        if FORCE_PYTHON_ROLLING_FACTORS:
            raise RuntimeError(
                "已启用逐交易日 Python 滚动，避免 DAI 对 date IN 的窗口谓词下推"
            )
        frame = query_df(sql, filters={"date": [moneyflow_query_start, END_DATE]})
        if frame is None or frame.empty:
            raise ValueError("SQL窗口计算没有返回调仓截面")
        source = "cn_stock_moneyflow.outflow_amount_main / SQL滚动10日求和"
    except Exception as sql_error:
        progress(
            "mfd_sellamt_d(10日累计)的DAI窗口方案未返回有效截面，"
            f"改用原始outflow_amount_main在Python中滚动计算：{sql_error}"
        )
        raw_candidates = [
            (
                f"""
                SELECT date, instrument, outflow_amount_main
                FROM cn_stock_moneyflow
                WHERE date >= '{moneyflow_query_start}' AND date <= '{END_DATE}'
                ORDER BY instrument, date
                """,
                {"date": [moneyflow_query_start, END_DATE]},
                "Python兜底/显式日期区间",
            ),
            (
                """
                SELECT date, instrument, outflow_amount_main
                FROM cn_stock_moneyflow
                ORDER BY instrument, date
                """,
                {"date": [moneyflow_query_start, END_DATE]},
                "Python兜底/DAI filters日期区间",
            ),
        ]
        raw, fallback_source = first_success_query(raw_candidates)
        raw["date"] = pd.to_datetime(raw["date"]).dt.normalize()
        raw["instrument"] = raw["instrument"].astype(str)
        raw["outflow_amount_main"] = pd.to_numeric(
            raw["outflow_amount_main"], errors="coerce", downcast="float"
        )
        raw = raw.dropna(subset=["date", "instrument"]).sort_values(
            ["instrument", "date"], kind="mergesort"
        ).reset_index(drop=True)
        raw["outflow_sum"] = (
            raw.groupby("instrument", sort=False)["outflow_amount_main"]
            .rolling(MONEYFLOW_WINDOW_DAYS, min_periods=MONEYFLOW_WINDOW_DAYS)
            .sum()
            .reset_index(level=0, drop=True)
        )
        raw["mfd_sellamt_d_sum_10"] = -raw["outflow_sum"]
        signal_set = set(pd.to_datetime(signal_dates))
        frame = raw.loc[
            raw["date"].isin(signal_set),
            ["date", "instrument", "mfd_sellamt_d_sum_10"],
        ].copy()
        del raw
        gc.collect()
        source = f"cn_stock_moneyflow.outflow_amount_main / {fallback_source}"
    progress(f"mfd_sellamt_d(10日累计)构建来源：{source}；公式=-rolling_sum(10)")
    return clean_factor_frame(frame, "mfd_sellamt_d_sum_10")


def clean_factor_frame(frame: pd.DataFrame, factor_name: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        raise ValueError(f"{factor_name}由原始字段计算后没有返回任何截面行。")
    out = frame[["date", "instrument", factor_name]].copy()
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out[factor_name] = pd.to_numeric(out[factor_name], errors="coerce")
    out = out.dropna(subset=["date", "instrument", factor_name])
    out = out[np.isfinite(out[factor_name])]
    out = out.drop_duplicates(["date", "instrument"], keep="last")
    if out.empty:
        raise ValueError(f"{factor_name}由原始字段计算后没有任何有限因子值。")
    return out


def build_raw_factor_panels(
    base_panel: pd.DataFrame,
    schedules: Mapping[str, dict],
    query_start: str,
) -> Tuple[Dict[str, pd.DataFrame], dict]:
    factor_dates = {}
    for cluster_name, config in _ACTIVE_CONFIGS.items():
        for factor_name in config["factor_weights"]:
            factor_dates.setdefault(factor_name, set()).update(schedules[cluster_name]["signal_dates"])

    panels: Dict[str, pd.DataFrame] = {}
    source_audit = {}
    if "hml_r_std_5m" in factor_dates:
        panels["hml_r_std_5m"] = query_hml_factor(sorted(factor_dates["hml_r_std_5m"]), query_start)
        source_audit["hml_r_std_5m"] = (
            "cn_stock_bar1d.high/low/pre_close；"
            "std(high/pre_close-1,105)-std(low/pre_close-1,105)"
        )
    if "exp_wgt_return_6m" in factor_dates:
        panels["exp_wgt_return_6m"] = query_exp_factor(sorted(factor_dates["exp_wgt_return_6m"]), query_start)
        source_audit["exp_wgt_return_6m"] = (
            "cn_stock_prefactors.close/turn；126日成交量加权指数衰减收益"
        )
    if "bias_std_turn_5d" in factor_dates:
        panels["bias_std_turn_5d"] = query_bias_factor(sorted(factor_dates["bias_std_turn_5d"]), query_start)
        source_audit["bias_std_turn_5d"] = (
            "cn_stock_prefactors.turn；std(turn>0,5)/std(turn>0,504)-1"
        )

    if "BP" in factor_dates:
        panels["BP"], source = query_bp_factor(sorted(factor_dates["BP"]))
        source_audit["BP"] = f"{source}；BP=1/PB"

    if "Profit_G_q" in factor_dates:
        panels["Profit_G_q"], source = query_profit_factor(sorted(factor_dates["Profit_G_q"]))
        source_audit["Profit_G_q"] = source
        progress(f"Profit_G_q数据来源：{source}")

    if "qfa_roe" in factor_dates:
        tmp = base_panel[base_panel["date"].isin(pd.to_datetime(sorted(factor_dates["qfa_roe"])))]
        tmp = tmp[["date", "instrument", "roe_avg_mrq"]].rename(columns={"roe_avg_mrq": "qfa_roe"})
        panels["qfa_roe"] = clean_factor_frame(tmp, "qfa_roe")
        source_audit["qfa_roe"] = "cn_stock_prefactors.roe_avg_mrq"
        progress("qfa_roe构建来源：cn_stock_prefactors.roe_avg_mrq（单季度平均ROE原始字段）")

    if "mfd_sellamt_d_sum_10" in factor_dates:
        panels["mfd_sellamt_d_sum_10"] = query_moneyflow_factor(
            sorted(factor_dates["mfd_sellamt_d_sum_10"]), query_start
        )
        source_audit["mfd_sellamt_d_sum_10"] = (
            "cn_stock_moneyflow.outflow_amount_main；-rolling_sum(10)"
        )

    return panels, source_audit


# =============================================================================
# 7. 因子预处理、组内线性合成与选股
# =============================================================================

def eligible_cross_section(base_date: pd.DataFrame) -> pd.DataFrame:
    frame = base_date.copy()
    frame = frame[frame["instrument"].map(board_allowed)]
    frame = frame[
        (frame["total_market_cap"] > 0)
        & frame["total_market_cap"].notna()
        & (frame["st_status"] == 0)
        & (frame["suspended"] == 0)
        & (frame["list_days"].fillna(0) >= MIN_LIST_DAYS)
    ]
    return frame.drop_duplicates("instrument", keep="last")


def transform_factor_cross_section(
    frame: pd.DataFrame,
    factor_name: str,
    spec: FactorSpec,
) -> pd.Series:
    initial_z = robust_zscore(frame[factor_name])
    processed = (
        neutralize_score(initial_z, frame["total_market_cap"], frame["industry"])
        if spec.neutralize
        else initial_z
    )
    return processed * int(spec.direction)


def select_cluster_on_date(
    cluster_name: str,
    signal_date: str,
    base_panel: pd.DataFrame,
    factor_panels: Mapping[str, pd.DataFrame],
    config: dict,
) -> Tuple[List[str], List[dict], pd.DataFrame]:
    date_value = pd.Timestamp(signal_date)
    base = eligible_cross_section(base_panel[base_panel["date"] == date_value])
    if base.empty:
        return [], [{"cluster": cluster_name, "signal_date": signal_date, "error": "基础股票池为空"}], pd.DataFrame()

    grouped = assign_market_cap_groups(base)
    if grouped.empty:
        return [], [{"cluster": cluster_name, "signal_date": signal_date, "error": "市值分组失败"}], pd.DataFrame()

    needed_factors = list(config["factor_weights"])
    work = grouped
    for factor_name in needed_factors:
        panel = factor_panels[factor_name]
        section = panel[panel["date"] == date_value][["instrument", factor_name]]
        work = work.merge(section, on="instrument", how="left", validate="one_to_one")

    for factor_name in needed_factors:
        work[f"{factor_name}__aligned"] = transform_factor_cross_section(
            work, factor_name, FACTOR_SPECS[factor_name]
        )

    audit_rows = []
    selected_parts = []
    for group_number in config["market_cap_groups"]:
        group_all = work[work["market_cap_group"] == group_number].copy()
        aligned_cols = [f"{name}__aligned" for name in needed_factors]
        group_valid = group_all.dropna(subset=aligned_cols).copy()
        for factor_name in needed_factors:
            # 因子先在市值组内转为排名百分位，再做线性组合。
            group_valid[f"{factor_name}__rank"] = group_valid[f"{factor_name}__aligned"].rank(
                method="average", pct=True, ascending=True
            )
        group_valid["composite_score"] = 0.0
        for factor_name, weight in config["factor_weights"].items():
            group_valid["composite_score"] += float(weight) * group_valid[f"{factor_name}__rank"]

        n_valid = len(group_valid)
        n_select = max(1, int(math.ceil(n_valid * float(config["select_pct"])))) if n_valid else 0
        selected = group_valid.sort_values(
            ["composite_score", "instrument"], ascending=[False, True], kind="mergesort"
        ).head(n_select)
        selected_parts.append(selected)
        audit_rows.append(
            {
                "cluster": cluster_name,
                "signal_date": signal_date,
                "market_cap_group": int(group_number),
                "group_population": int(len(group_all)),
                "complete_factor_count": int(n_valid),
                "selected_count": int(len(selected)),
                "select_pct": float(config["select_pct"]),
                "factor_weights": dict(config["factor_weights"]),
            }
        )

    selected_frame = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()
    instruments = list(dict.fromkeys(selected_frame.get("instrument", pd.Series(dtype=str)).astype(str).tolist()))
    # 只保留入选股票的因子排名与综合分，避免保存所有截面导致内存膨胀。
    return instruments, audit_rows, selected_frame


def build_cluster_signals(
    base_panel: pd.DataFrame,
    factor_panels: Mapping[str, pd.DataFrame],
    schedules: Mapping[str, dict],
    configs: Mapping[str, dict],
):
    targets = {}
    selection_audit = []
    score_audit = {}
    for cluster_index, (cluster_name, config) in enumerate(configs.items(), 1):
        dates = schedules[cluster_name]["signal_dates"]
        targets[cluster_name] = {}
        for index, signal_date in enumerate(dates, 1):
            instruments, rows, scored = select_cluster_on_date(
                cluster_name, signal_date, base_panel, factor_panels, config
            )
            targets[cluster_name][signal_date] = instruments
            selection_audit.extend(rows)
            if not scored.empty:
                score_audit[(cluster_name, signal_date)] = scored
            if index == 1 or index == len(dates) or index % 20 == 0:
                progress(
                    f"策略簇{cluster_index}/4 {config['display_name']}："
                    f"{index}/{len(dates)}，{signal_date}，入选{len(instruments)}只"
                )
    return targets, pd.DataFrame(selection_audit), score_audit


def summarize_factor_panels(factor_panels: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    """汇总每个因子的实际截面覆盖，优化模式和单次回测共用。"""
    rows = []
    for factor_name, panel in factor_panels.items():
        rows.append(
            {
                "factor": factor_name,
                "rows": int(len(panel)),
                "cross_sections": int(panel["date"].nunique()),
                "start_date": date_text(panel["date"].min()),
                "end_date": date_text(panel["date"].max()),
            }
        )
    return pd.DataFrame(rows).sort_values("factor", kind="mergesort").reset_index(drop=True)


def validate_initial_factor_warmup(
    base_panel: pd.DataFrame,
    factor_panels: Mapping[str, pd.DataFrame],
    schedules: Mapping[str, dict],
    configs: Mapping[str, dict],
) -> pd.DataFrame:
    """验收首个正式调仓截面，禁止因子预热失败后静默空仓。"""
    audit_rows = []
    failures = []
    for cluster_name, config in configs.items():
        signal_dates = schedules[cluster_name]["signal_dates"]
        if not signal_dates:
            failures.append(f"{cluster_name}: 没有正式信号日")
            continue
        first_signal = signal_dates[0]
        instruments, group_rows, _ = select_cluster_on_date(
            cluster_name, first_signal, base_panel, factor_panels, config
        )
        rows_by_group = {
            int(row["market_cap_group"]): row
            for row in group_rows
            if "market_cap_group" in row
        }
        for group_number in config["market_cap_groups"]:
            row = rows_by_group.get(int(group_number), {})
            population = int(row.get("group_population", 0) or 0)
            complete = int(row.get("complete_factor_count", 0) or 0)
            selected = int(row.get("selected_count", 0) or 0)
            audit_rows.append(
                {
                    "cluster": cluster_name,
                    "display_name": config["display_name"],
                    "first_signal_date": first_signal,
                    "market_cap_group": int(group_number),
                    "group_population": population,
                    "complete_factor_count": complete,
                    "selected_count": selected,
                    "complete_coverage": complete / population if population > 0 else np.nan,
                }
            )
            if complete < MIN_INITIAL_COMPLETE_STOCKS_PER_GROUP or selected <= 0:
                failures.append(
                    f"{config['display_name']} 首日{first_signal} 市值组{group_number}: "
                    f"完整因子{complete}/{population}，入选{selected}"
                )
        progress(
            f"预热验收：{config['display_name']}首日{first_signal}，"
            f"完整目标股票{len(instruments)}只"
        )

    audit = pd.DataFrame(audit_rows)
    if failures and STRICT_FACTOR_WARMUP_VALIDATION:
        detail = "\n".join(failures[:20])
        raise ValueError(
            "因子预热验收失败，已停止回测，避免策略簇在早期静默持有现金：\n" + detail
        )
    return audit


def delay_schedules_until_factor_ready(
    base_panel: pd.DataFrame,
    factor_panels: Mapping[str, pd.DataFrame],
    schedules: Mapping[str, dict],
    configs: Mapping[str, dict],
) -> Tuple[Dict[str, dict], pd.DataFrame]:
    """将每个策略簇的首个信号日推迟到其全部市值组均有足够完整因子的日期。

    不能因为某个长周期因子或数据表的可用历史较短，就用缺失值填充、
    降低样本门槛或关闭预热校验。这里仅跳过当日不可形成完整截面信号的
    调仓点；后续信号仍严格只使用该日及以前的数据。
    """
    adjusted = deepcopy(dict(schedules))
    audit_rows = []
    failures = []

    for cluster_name, config in configs.items():
        original_dates = list(schedules[cluster_name]["signal_dates"])
        first_ready_index = None
        first_ready_counts = {}

        for index, signal_date in enumerate(original_dates):
            _, group_rows, _ = select_cluster_on_date(
                cluster_name, signal_date, base_panel, factor_panels, config
            )
            rows_by_group = {
                int(row["market_cap_group"]): row
                for row in group_rows
                if "market_cap_group" in row
            }
            group_counts = {
                int(group_number): int(
                    rows_by_group.get(int(group_number), {}).get("complete_factor_count", 0) or 0
                )
                for group_number in config["market_cap_groups"]
            }
            group_selected = {
                int(group_number): int(
                    rows_by_group.get(int(group_number), {}).get("selected_count", 0) or 0
                )
                for group_number in config["market_cap_groups"]
            }
            ready = all(
                group_counts[group_number] >= MIN_INITIAL_COMPLETE_STOCKS_PER_GROUP
                and group_selected[group_number] > 0
                for group_number in group_counts
            )
            if ready:
                first_ready_index = index
                first_ready_counts = group_counts
                break

        if first_ready_index is None:
            failures.append(
                f"{config['display_name']}：在 {original_dates[0] if original_dates else START_DATE} 至 "
                f"{original_dates[-1] if original_dates else END_DATE} 没有满足完整因子门槛的调仓截面"
            )
            continue

        ready_dates = original_dates[first_ready_index:]
        adjusted[cluster_name]["signal_dates"] = ready_dates
        adjusted[cluster_name]["expected_trade_dates"] = {
            date: schedules[cluster_name]["expected_trade_dates"][date]
            for date in ready_dates
        }
        skipped_dates = original_dates[:first_ready_index]
        audit_rows.append(
            {
                "cluster": cluster_name,
                "display_name": config["display_name"],
                "original_first_signal_date": original_dates[0] if original_dates else None,
                "first_ready_signal_date": ready_dates[0],
                "skipped_signal_count": len(skipped_dates),
                "first_ready_complete_counts": dict(first_ready_counts),
            }
        )
        if skipped_dates:
            progress(
                f"因子可用性：{config['display_name']}跳过{len(skipped_dates)}个不完整信号日，"
                f"首个可交易截面为{ready_dates[0]}，各市值组完整样本={first_ready_counts}"
            )

    if failures:
        raise ValueError("因子可用性检查失败，无法生成无缺失的策略簇信号：\n" + "\n".join(failures))
    return adjusted, pd.DataFrame(audit_rows)


# =============================================================================
# 8. 趋势基线与防御覆盖层
# =============================================================================

def index_code_candidates(code: str) -> List[str]:
    code = str(code)
    number = code.split(".")[0]
    values = [code]
    for suffix in ("SH", "CSI", "XSHG"):
        candidate = f"{number}.{suffix}"
        if candidate not in values:
            values.append(candidate)
    return values


def build_trend_panel(config: dict, trade_dates: Sequence[pd.Timestamp]) -> pd.DataFrame:
    window = int(config["trend_ma_window"])
    query_start = date_text(pd.Timestamp(START_DATE) - timedelta(days=max(180, window * 4)))
    candidates = []
    for index_code in index_code_candidates(config["trend_index"]):
        safe_code = str(index_code).replace("'", "''")
        sql = f"""
        SELECT date, instrument, close
        FROM cn_stock_index_bar1d
        WHERE instrument = '{safe_code}'
        ORDER BY date
        """
        candidates.append((sql, {"date": [query_start, END_DATE]}, index_code))
    frame, actual_code = first_success_query(candidates)
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame["close"] = pd.to_numeric(frame["close"], errors="coerce")
    frame = frame.dropna(subset=["date", "close"]).drop_duplicates("date", keep="last").sort_values("date")
    frame["ma"] = frame["close"].rolling(window, min_periods=window).mean()
    frame["risk_on"] = np.where(frame["ma"].notna(), frame["close"] >= frame["ma"], True)
    frame = frame.set_index("date").reindex(pd.DatetimeIndex(trade_dates)).ffill()
    frame["risk_on"] = frame["risk_on"].fillna(True).astype(bool)
    frame["trend_index_requested"] = config["trend_index"]
    frame["trend_index_actual"] = actual_code
    return frame.reset_index().rename(columns={"index": "date"})


def build_all_trends(configs: Mapping[str, dict], trade_dates: Sequence[pd.Timestamp]):
    trends = {}
    for cluster_name, config in configs.items():
        if config.get("use_defensive_compensation", False):
            trends[cluster_name] = build_trend_panel(config, trade_dates)
            progress(
                f"{config['display_name']}趋势基线：{config['trend_index']} "
                f"MA{config['trend_ma_window']}"
            )
    return trends


def trend_state_for_date(cluster_name: str, today: str) -> bool:
    trend_lookup = _RUNTIME_DATA["trend_lookup"].get(cluster_name, {})
    row = trend_lookup.get(today)
    return bool(row["risk_on"]) if row is not None else True


def build_sleeve_component_weights(context, today: str):
    all_components = {}
    for cluster_name, config in _ACTIVE_CONFIGS.items():
        active_capital_weights = getattr(context, "active_cluster_capital_weights", {})
        capital_weight = float(active_capital_weights.get(cluster_name, config["capital_weight"]))
        instruments = list(dict.fromkeys(context.current_cluster_targets.get(cluster_name, [])))
        use_defense = bool(config.get("use_defensive_compensation", False))
        risk_on = trend_state_for_date(cluster_name, today) if use_defense else True

        if use_defense:
            factor_exposure = float(
                config["risk_on_factor_exposure"] if risk_on else config["risk_off_factor_exposure"]
            )
            defensive_exposure = 0.0 if risk_on else float(config["risk_off_defensive_exposure"])
            defensive_assets = tuple(str(x) for x in config["defensive_assets"])
        else:
            factor_exposure = float(config.get("normal_factor_exposure", DEFAULT_NORMAL_FACTOR_EXPOSURE))
            defensive_exposure = 0.0
            defensive_assets = ()

        factor_targets = {}
        if instruments:
            each = capital_weight * factor_exposure / len(instruments)
            factor_targets = {instrument: each for instrument in instruments}
        defensive_targets = {}
        # 防御覆盖层由趋势状态决定，不应因当期因子目标为空而被一并关闭。
        if defensive_assets and defensive_exposure > 0:
            each = capital_weight * defensive_exposure / len(defensive_assets)
            defensive_targets = {instrument: each for instrument in defensive_assets}

        all_components[cluster_name] = {
            "factor": factor_targets,
            "defensive": defensive_targets,
            "capital_weight": capital_weight,
            "risk_on": risk_on,
            "factor_exposure_inside_sleeve": factor_exposure if instruments else 0.0,
            "defensive_exposure_inside_sleeve": defensive_exposure if instruments else 0.0,
        }
    return all_components


def aggregate_component_weights(components: Mapping[str, dict]) -> Dict[str, float]:
    target = {}
    for sleeve in components.values():
        for component_name in ("factor", "defensive"):
            for instrument, weight in sleeve[component_name].items():
                target[instrument] = target.get(instrument, 0.0) + float(weight)
    return {instrument: weight for instrument, weight in target.items() if weight > 1e-12}


def ownership_shares_from_components(components: Mapping[str, dict]) -> Dict[str, List[Tuple[str, str, float]]]:
    """把每只股票的目标权重拆成策略簇/因子或防御组件的归属比例。"""
    raw: Dict[str, List[Tuple[str, str, float]]] = {}
    for cluster_name, sleeve in components.items():
        for component_name in ("factor", "defensive"):
            for instrument, weight in sleeve[component_name].items():
                if float(weight) > 0:
                    raw.setdefault(str(instrument), []).append(
                        (cluster_name, component_name, float(weight))
                    )
    shares = {}
    for instrument, rows in raw.items():
        total = sum(row[2] for row in rows)
        if total > 0:
            shares[instrument] = [
                (cluster_name, component_name, weight / total)
                for cluster_name, component_name, weight in rows
            ]
    return shares


# =============================================================================
# 9. 四个虚拟账户的资金使用归因
# =============================================================================

def record_capital_usage(context, today: str) -> None:
    components = getattr(context, "last_sleeve_components", {})
    pv = portfolio_value(context)
    positions = current_positions(context)

    actual = {
        cluster_name: {"factor": 0.0, "defensive": 0.0}
        for cluster_name in _ACTIVE_CONFIGS
    }
    live_position_instruments = set()
    for instrument, position in positions.items():
        if position_amount(position) <= 0:
            continue
        instrument = str(instrument)
        live_position_instruments.add(instrument)
        market_value = position_market_value(position)
        ownership_rows = context.ownership_shares.get(instrument, [])
        if not np.isfinite(market_value) or market_value <= 0 or not ownership_rows:
            continue
        for cluster_name, component_name, share in ownership_rows:
            actual[cluster_name][component_name] += market_value * float(share)

    # 已经完全卖出的旧股票不再保留归属记录；被停牌/跌停阻塞的旧持仓继续归属于原策略簇。
    current_target_instruments = set(aggregate_component_weights(components))
    stale = set(context.ownership_shares) - live_position_instruments - current_target_instruments
    for instrument in stale:
        context.ownership_shares.pop(instrument, None)

    for cluster_name, config in _ACTIVE_CONFIGS.items():
        sleeve = components.get(
            cluster_name,
            {"factor": {}, "defensive": {}, "capital_weight": float(config["capital_weight"]), "risk_on": True},
        )
        allocated = pv * float(config["capital_weight"])
        target_factor = pv * sum(sleeve["factor"].values())
        target_defensive = pv * sum(sleeve["defensive"].values())
        target_cash = allocated - target_factor - target_defensive
        actual_factor = actual[cluster_name]["factor"]
        actual_defensive = actual[cluster_name]["defensive"]
        actual_invested = actual_factor + actual_defensive
        context.capital_usage_records.append(
            {
                "date": today,
                "cluster": cluster_name,
                "display_name": config["display_name"],
                "portfolio_value": pv,
                "allocated_capital": allocated,
                "target_factor_value": target_factor,
                "target_defensive_value": target_defensive,
                "target_cash_value": target_cash,
                "actual_factor_value": actual_factor,
                "actual_defensive_value": actual_defensive,
                "actual_invested_value": actual_invested,
                "actual_unallocated_value": allocated - actual_invested,
                "actual_utilization": actual_invested / allocated if allocated > 0 else np.nan,
                "risk_on": bool(sleeve.get("risk_on", True)),
            }
        )


# =============================================================================
# 9A. 成交级策略簇账本：不再按“当前目标股票”覆盖历史归属
# =============================================================================

def component_weights_by_cluster(components: Mapping[str, dict]) -> Dict[str, Dict[str, float]]:
    result = {}
    for cluster_name, sleeve in components.items():
        weights = {}
        for component_name in ("factor", "defensive"):
            for instrument, weight in sleeve.get(component_name, {}).items():
                weights[str(instrument)] = weights.get(str(instrument), 0.0) + float(weight)
        result[cluster_name] = weights
    return result


def register_pending_cluster_trades(context, previous_components: Mapping[str, dict], components: Mapping[str, dict]) -> None:
    """记录每个策略簇对每只股票的目标变动；成交回调据此将合并成交拆回各簇。"""
    previous = component_weights_by_cluster(previous_components)
    current = component_weights_by_cluster(components)
    pending = {}
    for cluster_name in _ACTIVE_CONFIGS:
        instruments = set(previous.get(cluster_name, {})) | set(current.get(cluster_name, {}))
        for instrument in instruments:
            delta = float(current.get(cluster_name, {}).get(instrument, 0.0)) - float(previous.get(cluster_name, {}).get(instrument, 0.0))
            if abs(delta) > 1e-12:
                pending.setdefault(str(instrument), []).append({"cluster": cluster_name, "weight_delta": delta})
    context.pending_cluster_trade_requests = pending


def infer_trade_direction(trade, pending_rows: Sequence[Mapping[str, float]]) -> int:
    raw = " ".join(str(getattr(trade, field, "")) for field in ("side", "direction", "order_type", "entrust_bs")).lower()
    if any(token in raw for token in ("sell", "short", "卖", "s")):
        return -1
    if any(token in raw for token in ("buy", "cover", "买", "b")):
        return 1
    net = sum(float(row.get("weight_delta", 0.0)) for row in pending_rows)
    return 1 if net >= 0 else -1


def allocate_trade_to_cluster_ledger(context, trade) -> None:
    instrument = str(getattr(trade, "instrument", ""))
    quantity = abs(float(getattr(trade, "filled_qty", 0.0) or 0.0))
    if not instrument or not np.isfinite(quantity) or quantity <= 0:
        return
    pending_rows = list(getattr(context, "pending_cluster_trade_requests", {}).get(instrument, []))
    direction = infer_trade_direction(trade, pending_rows)
    requested = [row for row in pending_rows if float(row.get("weight_delta", 0.0)) * direction > 1e-12]
    if not requested:
        context.unmatched_trade_records.append({
            "trade_date": str(getattr(trade, "trade_date", "")), "instrument": instrument,
            "filled_qty": quantity, "direction": direction, "原因": "未找到同方向策略簇目标变动",
        })
        return
    total_request = sum(abs(float(row["weight_delta"])) for row in requested)
    money = abs(float(getattr(trade, "filled_money", 0.0) or 0.0))
    price = float(getattr(trade, "filled_price", 0.0) or 0.0)
    if money <= 0 and price > 0:
        money = quantity * price
    ledger = context.cluster_position_qty
    for row in requested:
        cluster_name = str(row["cluster"])
        share = abs(float(row["weight_delta"])) / total_request if total_request > 0 else 0.0
        allocated_qty = quantity * share
        previous_qty = float(ledger[cluster_name].get(instrument, 0.0))
        if direction < 0:
            allocated_qty = min(allocated_qty, max(0.0, previous_qty))
        ledger[cluster_name][instrument] = max(0.0, previous_qty + direction * allocated_qty)
        allocated_money = money * share
        if direction > 0:
            context.cluster_ledger_cash[cluster_name] -= allocated_money * (1.0 + BUY_COST)
        else:
            context.cluster_ledger_cash[cluster_name] += allocated_money * (1.0 - SELL_COST)
        context.cluster_trade_allocation_records.append({
            "trade_date": str(getattr(trade, "trade_date", "")), "instrument": instrument,
            "cluster": cluster_name, "display_name": _ACTIVE_CONFIGS[cluster_name]["display_name"],
            "direction": "买入" if direction > 0 else "卖出", "filled_qty": quantity,
            "allocated_qty": allocated_qty, "filled_money": money, "allocated_money": allocated_money,
            "request_weight_delta": float(row["weight_delta"]), "分摊来源": "目标变动比例",
        })


def reconcile_cluster_ledger(context, today: str) -> None:
    """将成交级数量账本与总账户真实持仓对齐；只把无法识别的残差明确记为对账调整。"""
    positions = current_positions(context)
    portfolio = portfolio_value(context)
    actual_position_value = 0.0
    per_cluster_value = {name: 0.0 for name in _ACTIVE_CONFIGS}
    active_weights = getattr(context, "active_cluster_capital_weights", {})
    components = getattr(context, "last_sleeve_components", {})
    target_weights = component_weights_by_cluster(components)
    for instrument, position in positions.items():
        instrument = str(instrument)
        actual_qty = max(0.0, position_amount(position))
        market_value = max(0.0, position_market_value(position))
        if actual_qty <= 0 or market_value <= 0:
            continue
        actual_position_value += market_value
        owned = {name: max(0.0, float(context.cluster_position_qty[name].get(instrument, 0.0))) for name in _ACTIVE_CONFIGS}
        owned_total = sum(owned.values())
        adjustment_type = "成交级账本"
        if owned_total <= 1e-12:
            fallback = {name: float(target_weights.get(name, {}).get(instrument, 0.0)) for name in _ACTIVE_CONFIGS}
            fallback_total = sum(fallback.values())
            if fallback_total <= 1e-12:
                fallback = {name: float(active_weights.get(name, 0.25)) for name in _ACTIVE_CONFIGS}
                fallback_total = sum(fallback.values())
            owned = {name: actual_qty * fallback[name] / fallback_total for name in _ACTIVE_CONFIGS}
            adjustment_type = "无成交回调的初始化/残差分摊"
        else:
            owned = {name: actual_qty * owned[name] / owned_total for name in _ACTIVE_CONFIGS}
            if abs(owned_total - actual_qty) > 1e-8:
                adjustment_type = "成交数量与实际持仓对账缩放"
        for cluster_name, qty in owned.items():
            context.cluster_position_qty[cluster_name][instrument] = qty
            value = market_value * qty / actual_qty
            per_cluster_value[cluster_name] += value
            context.cluster_position_records.append({
                "date": today, "instrument": instrument, "cluster": cluster_name,
                "display_name": _ACTIVE_CONFIGS[cluster_name]["display_name"],
                "实际总数量": actual_qty, "策略簇归属数量": qty, "实际市值": market_value,
                "策略簇归属市值": value, "对账方式": adjustment_type,
            })
    cash_total = max(0.0, portfolio - actual_position_value)
    positive_cash = {name: max(0.0, float(context.cluster_ledger_cash.get(name, 0.0))) for name in _ACTIVE_CONFIGS}
    cash_basis = sum(positive_cash.values())
    if cash_basis <= 1e-12:
        positive_cash = {name: float(active_weights.get(name, 0.25)) for name in _ACTIVE_CONFIGS}
        cash_basis = sum(positive_cash.values())
    allocated_cash = {name: cash_total * positive_cash[name] / cash_basis for name in _ACTIVE_CONFIGS}
    for cluster_name in _ACTIVE_CONFIGS:
        context.cluster_ledger_cash[cluster_name] = allocated_cash[cluster_name]
        sleeve_value = per_cluster_value[cluster_name] + allocated_cash[cluster_name]
        utilization = per_cluster_value[cluster_name] / sleeve_value if sleeve_value > 1e-12 else 0.0
        context.cluster_ledger_daily_records.append({
            "date": today, "cluster": cluster_name, "display_name": _ACTIVE_CONFIGS[cluster_name]["display_name"],
            "组合总资产": portfolio, "策略簇目标资金": portfolio * float(active_weights.get(cluster_name, 0.25)),
            "策略簇实际持仓市值": per_cluster_value[cluster_name], "策略簇实际现金": allocated_cash[cluster_name],
            "策略簇账本净值": sleeve_value, "实际资金利用率": utilization,
        })
    ledger_total = sum(per_cluster_value.values()) + sum(allocated_cash.values())
    # 供簇间再平衡器在下一次决策时使用：这里只保存已收盘、已成交并完成对账的实际簇净值，
    # 不再以目标股票权重或 ownership_shares 推断资金使用率。
    context.latest_sleeve_nav = {
        name: float(per_cluster_value[name] + allocated_cash[name]) for name in _ACTIVE_CONFIGS
    }
    context.latest_sleeve_weights = {
        name: value / ledger_total if ledger_total > 1e-12 else float(active_weights.get(name, 0.0))
        for name, value in context.latest_sleeve_nav.items()
    }
    context.cluster_reconciliation_records.append({
        "date": today, "组合总资产": portfolio, "总账户持仓市值": actual_position_value,
        "总账户现金": cash_total, "策略簇账本合计": ledger_total,
        "对账误差": ledger_total - portfolio,
        "对账误差比例": (ledger_total - portfolio) / portfolio if portfolio > 1e-12 else 0.0,
    })


# =============================================================================
# 10. BigTrader原生回测
# =============================================================================

def initialize(context: "bigtrader.IContext"):
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COMMISSION,
        )
    )
    # 明确指定“下一根日线开盘撮合”，确保 t 日收盘信号不会在 t 日开盘成交。
    try:
        if not hasattr(bigtrader, "VMatchAt"):
            raise AttributeError("当前 BigTrader 版本未提供 VMatchAt")
        next_match = getattr(
            bigtrader.VMatchAt,
            "NEXT_BAR_OPEN",
            getattr(bigtrader.VMatchAt, "NEXT", None),
        )
        if next_match is None:
            raise AttributeError("当前 BigTrader 版本未提供下一 Bar 开盘撮合枚举")
        context.set_vmatch_at(next_match)
    except Exception as exc:
        raise RuntimeError(f"无法设置下一交易日开盘撮合，已停止以避免前视偏差：{exc}") from exc
    try:
        context.set_stock_t1(1)
    except Exception:
        pass
    context.cluster_targets_by_signal_date = _RUNTIME_DATA["cluster_targets"]
    context.current_cluster_targets = {cluster_name: [] for cluster_name in _ACTIVE_CONFIGS}
    context.previous_risk_on = {cluster_name: None for cluster_name in _ACTIVE_CONFIGS}
    context.last_sleeve_components = {}
    context.last_aggregate_targets = {}
    # ownership_shares 仅保留兼容旧对象；实际资金归因完全改用成交级策略簇账本。
    context.ownership_shares = {}
    context.active_cluster_capital_weights = dict(_RUNTIME_DATA.get(
        "initial_cluster_capital_weights",
        {name: float(config["capital_weight"]) for name, config in _ACTIVE_CONFIGS.items()},
    ))
    context.cluster_position_qty = {name: {} for name in _ACTIVE_CONFIGS}
    context.cluster_ledger_cash = {
        name: CAPITAL_BASE * float(context.active_cluster_capital_weights.get(name, 0.25))
        for name in _ACTIVE_CONFIGS
    }
    context.pending_cluster_trade_requests = {}
    context.cluster_trade_allocation_records = RESEARCH_ARTIFACTS["cluster_trade_allocation_records"]
    context.cluster_ledger_daily_records = RESEARCH_ARTIFACTS["cluster_ledger_daily_records"]
    context.cluster_position_records = RESEARCH_ARTIFACTS["cluster_position_records"]
    context.cluster_reconciliation_records = RESEARCH_ARTIFACTS["cluster_reconciliation_records"]
    context.unmatched_trade_records = RESEARCH_ARTIFACTS["unmatched_trade_records"]
    context.capital_usage_records = RESEARCH_ARTIFACTS["capital_usage_records"]
    context.execution_audit = RESEARCH_ARTIFACTS["execution_audit"]
    context.order_event_audit = RESEARCH_ARTIFACTS["order_event_audit"]
    context.trade_event_audit = RESEARCH_ARTIFACTS["trade_event_audit"]
    context.expected_trade_dates = _RUNTIME_DATA["expected_trade_dates"]
    context.logger.info("四策略簇混合策略初始化完成。")


def submit_target_percent(context, instrument: str, target_weight: float):
    try:
        return_code = context.order_target_percent(str(instrument), float(target_weight))
        success = return_code is None or int(return_code) >= 0
        error = ""
        if not success and hasattr(context, "get_error_msg"):
            error = str(context.get_error_msg(int(return_code)))
        return success, return_code, error
    except Exception as exc:
        return False, None, str(exc)


def after_trading(context: "bigtrader.IContext", data: "bigtrader.IBarData"):
    """收盘后按成交级策略簇账本记录实际可见持仓并完成总账户对账。"""
    today = pd.Timestamp(data.current_dt).strftime("%Y-%m-%d")
    reconcile_cluster_ledger(context, today)


def handle_order(context, order):
    context.order_event_audit.append(
        {
            "instrument": str(getattr(order, "instrument", "")),
            "trading_day": str(getattr(order, "trading_day", "")),
            "order_status": str(getattr(order, "order_status", "")),
            "order_qty": getattr(order, "order_qty", np.nan),
            "filled_qty": getattr(order, "filled_qty", np.nan),
            "status_msg": str(getattr(order, "status_msg", "")),
        }
    )


def handle_trade(context, trade):
    context.trade_event_audit.append(
        {
            "instrument": str(getattr(trade, "instrument", "")),
            "trade_date": str(getattr(trade, "trade_date", "")),
            "filled_qty": getattr(trade, "filled_qty", np.nan),
            "filled_price": getattr(trade, "filled_price", np.nan),
            "filled_money": getattr(trade, "filled_money", np.nan),
        }
    )
    allocate_trade_to_cluster_ledger(context, trade)


# =============================================================================
# 11. 绘图与结果整理
# =============================================================================

_CHINESE_FONT_NAME = None
_CHINESE_FONT_CONFIGURED = False


def configure_chinese_font() -> Optional[str]:
    """使用BigQuant环境中实际存在的中文字体，避免只设置不存在的字体名称。"""
    global _CHINESE_FONT_NAME, _CHINESE_FONT_CONFIGURED
    if _CHINESE_FONT_CONFIGURED:
        return _CHINESE_FONT_NAME

    if CHINESE_FONT_PATH:
        font_path = os.path.abspath(os.path.expanduser(str(CHINESE_FONT_PATH)))
        if not os.path.isfile(font_path):
            raise FileNotFoundError(f"CHINESE_FONT_PATH不存在：{font_path}")
        font_manager.fontManager.addfont(font_path)
        _CHINESE_FONT_NAME = font_manager.FontProperties(fname=font_path).get_name()

    preferred_names = (
        "Noto Sans CJK SC", "Noto Sans CJK JP", "Source Han Sans CN",
        "Source Han Sans SC", "WenQuanYi Micro Hei", "WenQuanYi Zen Hei",
        "Microsoft YaHei", "SimHei", "PingFang SC", "Arial Unicode MS",
    )
    available_names = {entry.name for entry in font_manager.fontManager.ttflist}
    if _CHINESE_FONT_NAME is None:
        _CHINESE_FONT_NAME = next(
            (name for name in preferred_names if name in available_names), None
        )

    # 某些Linux镜像中的字体已安装但尚未进入Matplotlib缓存，按文件名再次探测并注册。
    if _CHINESE_FONT_NAME is None:
        path_keywords = (
            "notosanscjk", "sourcehansans", "wenquanyi", "wqy", "simhei",
            "msyh", "pingfang", "arialuni",
        )
        for font_path in font_manager.findSystemFonts(fontpaths=None, fontext="ttf"):
            normalized = os.path.basename(font_path).lower().replace("-", "").replace("_", "")
            if any(keyword in normalized for keyword in path_keywords):
                try:
                    font_manager.fontManager.addfont(font_path)
                    _CHINESE_FONT_NAME = font_manager.FontProperties(fname=font_path).get_name()
                    break
                except Exception:
                    continue

    plt.rcParams["font.family"] = "sans-serif"
    if _CHINESE_FONT_NAME:
        plt.rcParams["font.sans-serif"] = [_CHINESE_FONT_NAME, "DejaVu Sans"]
        progress(f"Matplotlib中文字体：{_CHINESE_FONT_NAME}")
    else:
        plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
        print(
            "警告：当前BigQuant环境未探测到中文字体。请安装Noto Sans CJK或设置CHINESE_FONT_PATH。",
            flush=True,
        )
    plt.rcParams["axes.unicode_minus"] = False
    _CHINESE_FONT_CONFIGURED = True
    return _CHINESE_FONT_NAME


def plot_capital_usage(capital_usage: pd.DataFrame) -> None:
    if capital_usage.empty:
        print("资金使用审计为空，跳过绘图。")
        return
    configure_chinese_font()
    cluster_order = list(_ACTIVE_CONFIGS)
    fig, axes = plt.subplots(len(cluster_order), 1, figsize=(15, 3.6 * len(cluster_order)), sharex=True)
    axes = np.atleast_1d(axes)
    for axis, cluster_name in zip(axes, cluster_order):
        frame = capital_usage[capital_usage["cluster"] == cluster_name].copy()
        frame["date"] = pd.to_datetime(frame["date"])
        frame = frame.sort_values("date")
        scale = 10_000.0
        factor = frame["target_factor_value"].clip(lower=0).to_numpy() / scale
        defensive = frame["target_defensive_value"].clip(lower=0).to_numpy() / scale
        cash = frame["target_cash_value"].clip(lower=0).to_numpy() / scale
        x = frame["date"].to_numpy()
        axis.stackplot(
            x,
            factor,
            defensive,
            cash,
            labels=["目标因子资金", "目标防御资金", "目标现金"],
            colors=["#4472C4", "#70AD47", "#D9D9D9"],
            alpha=0.82,
        )
        axis.plot(
            x,
            frame["actual_invested_value"].to_numpy() / scale,
            color="#C00000",
            linewidth=1.1,
            label="实际归因持仓市值",
        )
        axis.set_title(_ACTIVE_CONFIGS[cluster_name]["display_name"])
        axis.set_ylabel("资金（万元）")
        axis.grid(alpha=0.25)
        axis.legend(loc="upper left", ncol=4, fontsize=9)
    axes[-1].set_xlabel("日期")
    fig.suptitle("四个虚拟策略账户的独立资金使用情况", fontsize=16, y=0.995)
    fig.tight_layout()
    plt.show()

    utilization = capital_usage.pivot_table(
        index="date", columns="display_name", values="actual_utilization", aggfunc="last"
    ).sort_index()
    ax = utilization.plot(figsize=(15, 6), linewidth=1.2)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8)
    ax.set_title("各策略账户实际资金使用率")
    ax.set_xlabel("日期")
    ax.set_ylabel("实际持仓市值 / 分配资金")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def build_execution_exception_audit(capital_usage: pd.DataFrame) -> pd.DataFrame:
    """标记目标资金与实际归因持仓严重偏离的日期，供订单和成交明细复核。"""
    if capital_usage.empty:
        return pd.DataFrame()
    frame = capital_usage.copy()
    frame["actual_utilization"] = pd.to_numeric(frame["actual_utilization"], errors="coerce")
    frame["异常类型"] = np.where(
        frame["actual_utilization"] > 1.20,
        "实际归因持仓超过分配资金20%",
        np.where(frame["actual_utilization"] < 0.50, "实际归因持仓低于分配资金50%", ""),
    )
    total = frame.groupby("date", as_index=False).agg(
        组合总资产=("portfolio_value", "last"),
        各簇实际归因持仓合计=("actual_invested_value", "sum"),
    )
    total["组合级归因异常"] = total["各簇实际归因持仓合计"] > total["组合总资产"] * 1.01
    frame = frame.merge(total, on="date", how="left")
    audit = frame[(frame["异常类型"] != "") | frame["组合级归因异常"]].copy()
    if not audit.empty:
        audit = audit.sort_values(["date", "cluster"]).reset_index(drop=True)
    RESEARCH_ARTIFACTS["execution_exception_audit"] = audit
    RESEARCH_ARTIFACTS["execution_exception_output_path"] = save_result_table(
        audit, "执行与归因异常审计.csv"
    )
    return audit


def finalize_artifacts(performance):
    table_names = (
        "capital_usage_records", "execution_audit", "order_event_audit", "trade_event_audit"
    )
    for name in table_names:
        RESEARCH_ARTIFACTS[name.replace("_records", "")] = pd.DataFrame(RESEARCH_ARTIFACTS[name])
    RESEARCH_ARTIFACTS["performance"] = performance
    capital_usage = RESEARCH_ARTIFACTS["capital_usage"]
    if not capital_usage.empty:
        summary = (
            capital_usage.groupby(["cluster", "display_name"], as_index=False)
            .agg(
                average_allocated_capital=("allocated_capital", "mean"),
                average_actual_invested=("actual_invested_value", "mean"),
                average_actual_utilization=("actual_utilization", "mean"),
                minimum_actual_utilization=("actual_utilization", "min"),
                maximum_actual_utilization=("actual_utilization", "max"),
            )
        )
        RESEARCH_ARTIFACTS["capital_usage_summary"] = summary
        RESEARCH_ARTIFACTS["capital_usage_summary_output_path"] = save_result_table(
            summary, "各策略簇资金使用汇总.csv"
        )
        exception_audit = build_execution_exception_audit(capital_usage)
        print("\n各策略簇资金使用摘要：")
        display(summary)
        if not exception_audit.empty:
            print("\n执行与归因异常审计：")
            display(exception_audit.head(100))
        if PLOT_CAPITAL_USAGE:
            plot_capital_usage(capital_usage)

    ledger_daily = pd.DataFrame(RESEARCH_ARTIFACTS.get("cluster_ledger_daily_records", []))
    ledger_positions = pd.DataFrame(RESEARCH_ARTIFACTS.get("cluster_position_records", []))
    ledger_trades = pd.DataFrame(RESEARCH_ARTIFACTS.get("cluster_trade_allocation_records", []))
    reconciliation = pd.DataFrame(RESEARCH_ARTIFACTS.get("cluster_reconciliation_records", []))
    unmatched = pd.DataFrame(RESEARCH_ARTIFACTS.get("unmatched_trade_records", []))
    RESEARCH_ARTIFACTS["cluster_ledger_daily"] = ledger_daily
    RESEARCH_ARTIFACTS["cluster_ledger_positions"] = ledger_positions
    RESEARCH_ARTIFACTS["cluster_ledger_trades"] = ledger_trades
    RESEARCH_ARTIFACTS["cluster_ledger_reconciliation"] = reconciliation
    if not ledger_daily.empty:
        RESEARCH_ARTIFACTS["cluster_ledger_output_path"] = save_result_table(ledger_daily, "策略簇持仓账本.csv")
    if not ledger_positions.empty:
        RESEARCH_ARTIFACTS["cluster_positions_output_path"] = save_result_table(ledger_positions, "策略簇持仓归属明细.csv")
    if not ledger_trades.empty:
        RESEARCH_ARTIFACTS["cluster_trades_output_path"] = save_result_table(ledger_trades, "策略簇成交分摊明细.csv")
    if not reconciliation.empty:
        RESEARCH_ARTIFACTS["cluster_reconciliation_output_path"] = save_result_table(reconciliation, "资金分配账本对账表.csv")
        max_error = float(pd.to_numeric(reconciliation["对账误差比例"], errors="coerce").abs().max())
        if np.isfinite(max_error) and max_error > 0.001:
            warnings.warn(f"策略簇账本最大对账误差为{max_error:.4%}，请检查成交回调字段。")
    if not unmatched.empty:
        RESEARCH_ARTIFACTS["unmatched_trade_output_path"] = save_result_table(unmatched, "未匹配成交审计.csv")


# =============================================================================
# 12. Optuna缓存、快速回测与参数搜索
# =============================================================================

def optimization_union_schedules(trade_dates: Sequence[pd.Timestamp]) -> dict:
    """为每个策略簇汇总所有候选调仓周期会用到的信号日期。"""
    schedules = {}
    next_map = {
        date_text(trade_dates[i]): date_text(trade_dates[i + 1])
        for i in range(len(trade_dates) - 1)
    }
    for cluster_name, space in OPTIMIZATION_SPACE.items():
        union_dates = set()
        for n in space["rebalance_days"]:
            union_dates.update(
                date_text(value)
                for value in trade_dates[::int(n)]
                if date_text(value) in next_map
            )
        ordered = sorted(union_dates)
        schedules[cluster_name] = {
            "signal_dates": ordered,
            "expected_trade_dates": {value: next_map[value] for value in ordered},
        }
    return schedules, next_map


def build_optimization_rank_cache(
    base_panel: pd.DataFrame,
    factor_panels: Mapping[str, pd.DataFrame],
    schedules: Mapping[str, dict],
    configs: Mapping[str, dict],
) -> Dict[Tuple[str, str], pd.DataFrame]:
    """缓存方向统一和中性化后的组内排名；Trial只做加权与截取。"""
    cache: Dict[Tuple[str, str], pd.DataFrame] = {}
    total = sum(len(item["signal_dates"]) for item in schedules.values())
    completed = 0
    for cluster_name, config in configs.items():
        factor_names = list(config["factor_weights"])
        for signal_date in schedules[cluster_name]["signal_dates"]:
            date_value = pd.Timestamp(signal_date)
            base = eligible_cross_section(base_panel[base_panel["date"] == date_value])
            grouped = assign_market_cap_groups(base) if not base.empty else pd.DataFrame()
            if grouped.empty:
                cache[(cluster_name, signal_date)] = pd.DataFrame()
                completed += 1
                continue
            work = grouped
            for factor_name in factor_names:
                section = factor_panels[factor_name]
                section = section[section["date"] == date_value][["instrument", factor_name]]
                work = work.merge(section, on="instrument", how="left", validate="one_to_one")
            for factor_name in factor_names:
                work[f"{factor_name}__aligned"] = transform_factor_cross_section(
                    work, factor_name, FACTOR_SPECS[factor_name]
                )

            group_parts = []
            aligned_cols = [f"{name}__aligned" for name in factor_names]
            for group_number, group in work.groupby("market_cap_group", sort=True):
                valid = group.dropna(subset=aligned_cols).copy()
                if valid.empty:
                    continue
                for factor_name in factor_names:
                    valid[f"{factor_name}__rank"] = valid[f"{factor_name}__aligned"].rank(
                        method="average", pct=True, ascending=True
                    ).astype(np.float32)
                keep = ["instrument", "market_cap_group"] + [
                    f"{name}__rank" for name in factor_names
                ]
                group_parts.append(valid[keep])
            cache[(cluster_name, signal_date)] = (
                pd.concat(group_parts, ignore_index=True) if group_parts else pd.DataFrame()
            )
            completed += 1
            if completed == 1 or completed == total or completed % 50 == 0:
                progress(f"优化排名缓存：{completed}/{total}（{completed / total:.1%}）")
    return cache


def candidate_instruments_from_rank_cache(rank_cache: Mapping[Tuple[str, str], pd.DataFrame]) -> List[str]:
    allowed_groups = {
        cluster_name: {
            int(group)
            for option in space["market_cap_groups"]
            for group in option
        }
        for cluster_name, space in OPTIMIZATION_SPACE.items()
    }
    instruments = set(DEFAULT_DEFENSIVE_BANKS)
    for (cluster_name, _), frame in rank_cache.items():
        if frame is None or frame.empty:
            continue
        part = frame[frame["market_cap_group"].isin(allowed_groups[cluster_name])]
        instruments.update(part["instrument"].astype(str).tolist())
    return sorted(instruments)


def sql_string_list(values: Sequence[str]) -> str:
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


def query_fast_market_matrices(
    trade_dates: Sequence[pd.Timestamp], instruments: Sequence[str]
) -> dict:
    """一次读取优化候选股票的日开盘价及交易约束，构造紧凑矩阵。"""
    if not instruments:
        raise ValueError("优化候选股票列表为空。")
    start_text, end_text = date_text(min(trade_dates)), date_text(max(trade_dates))
    chunks = []
    chunk_size = 600
    for begin in range(0, len(instruments), chunk_size):
        chunk = list(instruments[begin:begin + chunk_size])
        sql = f"""
        SELECT date, instrument, open, upper_limit, lower_limit, suspended
        FROM cn_stock_factors_base
        WHERE date >= '{start_text}' AND date <= '{end_text}'
          AND instrument IN ({sql_string_list(chunk)})
        ORDER BY date, instrument
        """
        frame = query_df(sql, filters={"date": [start_text, end_text]})
        if frame is not None and not frame.empty:
            chunks.append(frame)
        progress(
            f"快速回测行情：{min(begin + chunk_size, len(instruments))}/{len(instruments)}只"
        )
    if not chunks:
        raise ValueError("快速回测行情查询为空。")
    market = pd.concat(chunks, ignore_index=True)
    market["date"] = pd.to_datetime(market["date"]).dt.normalize()
    market["instrument"] = market["instrument"].astype(str)
    for column in ("open", "upper_limit", "lower_limit", "suspended"):
        market[column] = pd.to_numeric(market[column], errors="coerce", downcast="float")
    market = market.drop_duplicates(["date", "instrument"], keep="last")
    valid_open = market["open"].notna() & (market["open"] > 0)
    valid_limit = (
        valid_open
        & market["upper_limit"].notna() & (market["upper_limit"] > 0)
        & market["lower_limit"].notna() & (market["lower_limit"] > 0)
    )
    not_suspended = market["suspended"].fillna(1).eq(0)
    market["can_buy"] = not_suspended & valid_open & ~(
        valid_limit & (market["open"] >= market["upper_limit"] * (1.0 - FAST_LIMIT_EPS))
    )
    market["can_sell"] = not_suspended & valid_open & ~(
        valid_limit & (market["open"] <= market["lower_limit"] * (1.0 + FAST_LIMIT_EPS))
    )

    date_index = pd.DatetimeIndex(trade_dates)
    columns = pd.Index(sorted(set(instruments)), dtype=object)
    open_matrix = market.pivot(index="date", columns="instrument", values="open")
    open_matrix = open_matrix.reindex(index=date_index, columns=columns).astype(np.float32)
    # 停牌期间价格保持不变；复牌后的首个开盘价会反映停牌期间累计跳变。
    open_filled = open_matrix.ffill()
    return_matrix = open_filled.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    return_matrix = return_matrix.fillna(0.0).astype(np.float32)
    can_buy = market.pivot(index="date", columns="instrument", values="can_buy")
    can_sell = market.pivot(index="date", columns="instrument", values="can_sell")
    can_buy = can_buy.reindex(index=date_index, columns=columns).fillna(False).astype(bool)
    can_sell = can_sell.reindex(index=date_index, columns=columns).fillna(False).astype(bool)
    del market, open_matrix, open_filled, chunks
    gc.collect()
    return {"returns": return_matrix, "can_buy": can_buy, "can_sell": can_sell}


def query_optimization_trend_close(trade_dates: Sequence[pd.Timestamp]) -> Dict[str, pd.Series]:
    result = {}
    max_window = max(
        max(space.get("trend_ma_window", (1,))) for space in OPTIMIZATION_SPACE.values()
    )
    query_start = date_text(pd.Timestamp(START_DATE) - timedelta(days=max(365, max_window * 4)))
    requested_codes = {
        config["trend_index"]
        for config in CLUSTER_CONFIGS.values()
        if config.get("use_defensive_compensation", False)
    }
    for requested in requested_codes:
        candidates = []
        for code in index_code_candidates(requested):
            safe = code.replace("'", "''")
            sql = f"""
            SELECT date, instrument, close
            FROM cn_stock_index_bar1d
            WHERE instrument = '{safe}'
            ORDER BY date
            """
            candidates.append((sql, {"date": [query_start, END_DATE]}, code))
        frame, actual = first_success_query(candidates)
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["close"] = pd.to_numeric(frame["close"], errors="coerce")
        series = frame.dropna(subset=["date", "close"]).drop_duplicates("date", keep="last")
        series = series.set_index("date")["close"].sort_index()
        # 保留回测开始日前的历史，使MA40/60/80/120在首个回测日已经充分预热。
        result[requested] = series
        progress(f"优化趋势缓存：{requested}，实际代码{actual}")
    return result


def prepare_optimization_cache(configs: Mapping[str, dict]) -> dict:
    """一次准备所有 trial 共用的数据，之后 trial 不再访问 DAI。

    缓存包含候选调仓日的点时股票池、因子面板、组内排名、开盘价收益、涨跌停/停牌约束和
    趋势指数收盘价。这样每个 trial 只变更参数与快速模拟结果，避免数据查询顺序影响比较。
    """
    progress("[优化1/6] 获取交易日历与候选调仓日期并集")
    trade_dates = query_trading_calendar()
    union_schedules, next_date_map = optimization_union_schedules(trade_dates)
    all_signal_dates = sorted({
        value for schedule in union_schedules.values() for value in schedule["signal_dates"]
    })
    progress(f"优化候选信号日期并集：{len(all_signal_dates)}个")

    progress("[优化2/6] 一次性读取候选截面的股票池和七个因子")
    base_panel = query_base_panel(all_signal_dates)
    query_start = query_factor_warmup_start()
    factor_panels, source_audit = build_raw_factor_panels(
        base_panel, union_schedules, query_start
    )
    factor_panel_summary = summarize_factor_panels(factor_panels)
    for row in factor_panel_summary.itertuples(index=False):
        progress(
            f"因子覆盖：{row.factor}，{row.start_date}至{row.end_date}，"
            f"{row.cross_sections}个截面、{row.rows:,}行"
        )

    # 优化模式要验收搜索空间可能使用的全部市值组，而不只验收默认配置。
    validation_configs = deepcopy(dict(configs))
    for cluster_name, space in OPTIMIZATION_SPACE.items():
        validation_configs[cluster_name]["market_cap_groups"] = sorted(
            {
                int(group)
                for option in space["market_cap_groups"]
                for group in option
            }
        )
    union_schedules, factor_availability_audit = delay_schedules_until_factor_ready(
        base_panel, factor_panels, union_schedules, validation_configs
    )
    warmup_audit = validate_initial_factor_warmup(
        base_panel, factor_panels, union_schedules, validation_configs
    )

    progress("[优化3/6] 缓存因子方向、中性化及市值组内排名")
    rank_cache = build_optimization_rank_cache(
        base_panel, factor_panels, union_schedules, configs
    )
    candidate_instruments = candidate_instruments_from_rank_cache(rank_cache)
    progress(f"优化候选股票并集：{len(candidate_instruments):,}只")

    progress("[优化4/6] 一次性读取快速回测行情和交易约束")
    market_matrices = query_fast_market_matrices(trade_dates, candidate_instruments)

    progress("[优化5/6] 缓存趋势指数收盘价")
    trend_close = query_optimization_trend_close(trade_dates)

    progress("[优化6/6] 优化缓存准备完成")
    return {
        "trade_dates": list(trade_dates),
        "union_schedules": union_schedules,
        "next_date_map": next_date_map,
        "base_panel": base_panel,
        "factor_panels": factor_panels,
        "factor_source_audit": source_audit,
        "factor_panel_summary": factor_panel_summary,
        "factor_availability_audit": factor_availability_audit,
        "factor_warmup_audit": warmup_audit,
        "factor_query_start": query_start,
        "rank_cache": rank_cache,
        "candidate_instruments": candidate_instruments,
        "market_matrices": market_matrices,
        "trend_close": trend_close,
    }


def normalized_trial_weights(trial, prefix: str, count: int, minimum: float = 0.0) -> List[float]:
    """将 TPE 抽样的正数归一化为和为 1 的权重，并可为每项预留最低权重。"""
    raw = np.array([
        trial.suggest_float(f"{prefix}_{i + 1}", 0.05, 1.0, log=True)
        for i in range(count)
    ], dtype=float)
    raw /= raw.sum()
    if minimum > 0:
        if minimum * count >= 1:
            raise ValueError("minimum * count必须小于1。")
        raw = minimum + (1.0 - minimum * count) * raw
    return raw.tolist()


def bounded_cluster_capital_weights(trial) -> List[float]:
    """生成四簇静态资金锚：合计 100%，每簇不低于下限，簇 1 不高于上限。"""
    weights = np.asarray(
        normalized_trial_weights(trial, "capital", 4, minimum=MIN_CLUSTER_CAPITAL_WEIGHT),
        dtype=float,
    )
    if weights[0] <= MAX_CLUSTER_1_CAPITAL_WEIGHT:
        return weights.tolist()
    remainder = 1.0 - MAX_CLUSTER_1_CAPITAL_WEIGHT
    other = weights[1:]
    other = other / other.sum()
    # 其余三簇至少各保留最低资金；剩余部分按 Trial 的相对偏好分配。
    other = MIN_CLUSTER_CAPITAL_WEIGHT + (
        remainder - 3 * MIN_CLUSTER_CAPITAL_WEIGHT
    ) * other
    return [MAX_CLUSTER_1_CAPITAL_WEIGHT, *other.tolist()]


def suggest_optimized_configs(trial) -> dict:
    """从一个 TPE trial 构造完整四簇配置。

    本函数只抽取允许搜索的内部参数和静态资金锚；簇间周期由
    run_joint_periodic_search 中的 periodic_rebalance_days 单独抽取。返回前统一校验权重、
    市值组、仓位暴露和四簇资金锚是否符合硬边界。
    """
    configs = deepcopy(CLUSTER_CONFIGS)
    c1 = normalized_trial_weights(trial, "c1_factor", 3)
    configs["cluster_1_defensive_small"]["factor_weights"] = dict(zip(
        ("hml_r_std_5m", "exp_wgt_return_6m", "bias_std_turn_5d"), c1
    ))
    c2 = normalized_trial_weights(trial, "c2_factor", 2)
    configs["cluster_2_value_growth"]["factor_weights"] = dict(zip(
        ("BP", "Profit_G_q"), c2
    ))

    capital = bounded_cluster_capital_weights(trial)
    for value, cluster_name in zip(capital, configs):
        configs[cluster_name]["capital_weight"] = value

    for cluster_name, space in OPTIMIZATION_SPACE.items():
        config = configs[cluster_name]
        group_labels = ["-".join(map(str, option)) for option in space["market_cap_groups"]]
        selected_label = trial.suggest_categorical(f"{cluster_name}_groups", group_labels)
        config["market_cap_groups"] = list(map(int, str(selected_label).split("-")))
        config["select_pct"] = float(trial.suggest_categorical(
            f"{cluster_name}_select_pct", list(space["select_pct"])
        ))
        config["rebalance_days"] = int(trial.suggest_categorical(
            f"{cluster_name}_rebalance_days", list(space["rebalance_days"])
        ))
        if config.get("use_defensive_compensation", False):
            config["trend_ma_window"] = int(trial.suggest_categorical(
                f"{cluster_name}_trend_ma_window", list(space["trend_ma_window"])
            ))
            risk_off = float(trial.suggest_categorical(
                f"{cluster_name}_risk_off_factor_exposure",
                list(space["risk_off_factor_exposure"]),
            ))
            risk_on = float(config["risk_on_factor_exposure"])
            config["risk_off_factor_exposure"] = risk_off
            config["risk_off_defensive_exposure"] = risk_on - risk_off
    return validate_configs(configs)


def targets_from_rank_cache(configs: Mapping[str, dict], cache: Mapping[str, object]) -> dict:
    schedules = cache.get("fixed_schedules")
    if schedules is None:
        schedules, _ = build_schedules(cache["trade_dates"], configs)
    targets = {cluster_name: {} for cluster_name in configs}
    rank_cache = cache["rank_cache"]
    for cluster_name, config in configs.items():
        factor_names = list(config["factor_weights"])
        rank_columns = [f"{name}__rank" for name in factor_names]
        for signal_date in schedules[cluster_name]["signal_dates"]:
            frame = rank_cache.get((cluster_name, signal_date))
            if frame is None or frame.empty:
                targets[cluster_name][signal_date] = []
                continue
            work = frame[frame["market_cap_group"].isin(config["market_cap_groups"])].copy()
            if work.empty:
                targets[cluster_name][signal_date] = []
                continue
            work["composite_score"] = 0.0
            for factor_name, weight in config["factor_weights"].items():
                work["composite_score"] += float(weight) * work[f"{factor_name}__rank"]
            selected_parts = []
            for _, group in work.groupby("market_cap_group", sort=True):
                n_select = max(1, int(math.ceil(len(group) * float(config["select_pct"]))))
                selected_parts.append(group.sort_values(
                    ["composite_score", "instrument"],
                    ascending=[False, True], kind="mergesort"
                ).head(n_select))
            selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()
            targets[cluster_name][signal_date] = (
                selected["instrument"].astype(str).drop_duplicates().tolist()
                if not selected.empty else []
            )
    return targets


def trend_states_for_configs(configs: Mapping[str, dict], cache: Mapping[str, object]) -> dict:
    states = {}
    trade_index = pd.DatetimeIndex(cache["trade_dates"])
    for cluster_name, config in configs.items():
        if not config.get("use_defensive_compensation", False):
            states[cluster_name] = pd.Series(True, index=trade_index)
            continue
        full_close = cache["trend_close"][config["trend_index"]].sort_index()
        window = int(config["trend_ma_window"])
        full_ma = full_close.rolling(window, min_periods=window).mean()
        close = full_close.reindex(trade_index).ffill()
        ma = full_ma.reindex(trade_index).ffill()
        states[cluster_name] = (close >= ma).where(ma.notna(), True).astype(bool)
    return states


def aggregate_fast_targets(
    configs: Mapping[str, dict], current_targets: Mapping[str, Sequence[str]], risk_on: Mapping[str, bool]
) -> Dict[str, float]:
    aggregate: Dict[str, float] = {}
    for cluster_name, config in configs.items():
        instruments = list(dict.fromkeys(map(str, current_targets.get(cluster_name, []))))
        capital_weight = float(config["capital_weight"])
        if config.get("use_defensive_compensation", False):
            is_on = bool(risk_on[cluster_name])
            factor_exposure = float(
                config["risk_on_factor_exposure"] if is_on else config["risk_off_factor_exposure"]
            )
            defensive_exposure = 0.0 if is_on else float(config["risk_off_defensive_exposure"])
        else:
            factor_exposure = float(config.get("normal_factor_exposure", DEFAULT_NORMAL_FACTOR_EXPOSURE))
            defensive_exposure = 0.0
        if instruments:
            each_factor = capital_weight * factor_exposure / len(instruments)
            for instrument in instruments:
                aggregate[instrument] = aggregate.get(instrument, 0.0) + each_factor
        # 与BigTrader原生路径保持一致：风险关闭时，即使因子目标暂时为空，
        # 防御资产仍按趋势覆盖层配置；未能使用的因子仓位保留为现金。
        if defensive_exposure > 0:
            assets = tuple(map(str, config["defensive_assets"]))
            each_defense = capital_weight * defensive_exposure / len(assets)
            for instrument in assets:
                aggregate[instrument] = aggregate.get(instrument, 0.0) + each_defense
    return aggregate


def build_fast_execution_targets(configs: Mapping[str, dict], cache: Mapping[str, object]) -> dict:
    """把 t 日收盘形成的簇内目标映射为 t+1 日开盘执行的股票目标权重。

    只有簇内选股目标变化或趋势状态切换时生成新目标；簇间周期再平衡不在此处处理，
    而是在四个独立影子账户收益汇总后由 simulate_joint_periodic_portfolio 单独计成本。
    """
    cluster_targets = targets_from_rank_cache(configs, cache)
    trend_states = trend_states_for_configs(configs, cache)
    next_date_map = cache["next_date_map"]
    current_targets = {name: [] for name in configs}
    previous_risk = {name: None for name in configs}
    execution_targets = {}
    for date_value in cache["trade_dates"]:
        today = date_text(date_value)
        changed = False
        risk_today = {}
        for cluster_name, config in configs.items():
            new_targets = cluster_targets[cluster_name].get(today)
            if new_targets is not None:
                current_targets[cluster_name] = list(new_targets)
                changed = True
            state = bool(trend_states[cluster_name].loc[pd.Timestamp(date_value)])
            risk_today[cluster_name] = state
            if config.get("use_defensive_compensation", False):
                if previous_risk[cluster_name] is None or previous_risk[cluster_name] != state:
                    changed = True
                previous_risk[cluster_name] = state
        if changed and any(current_targets.values()) and today in next_date_map:
            execution_targets[next_date_map[today]] = aggregate_fast_targets(
                configs, current_targets, risk_today
            )
    return execution_targets


def fast_portfolio_simulation(configs: Mapping[str, dict], cache: Mapping[str, object]) -> pd.DataFrame:
    """T 日收盘信号、T+1 日开盘执行的快速股票级模拟。

    日收益采用相邻开盘价收益：先结算持仓从上一开盘至当日开盘的涨跌，再按当日开盘的
    涨跌停/停牌约束执行调仓，并扣除买卖两侧成本。该模拟用于 TPE 排序；最终候选仍必须
    经过一次 BigTrader 原生回测确认。
    """
    execution_targets = build_fast_execution_targets(configs, cache)
    matrices = cache["market_matrices"]
    returns = matrices["returns"]
    can_buy = matrices["can_buy"]
    can_sell = matrices["can_sell"]
    current: Dict[str, float] = {}
    nav = 1.0
    rows = []
    for index, date_value in enumerate(cache["trade_dates"]):
        date_value = pd.Timestamp(date_value)
        start_nav = nav
        if index > 0 and current:
            day_returns = returns.loc[date_value]
            portfolio_return = sum(
                weight * float(day_returns.get(instrument, 0.0) or 0.0)
                for instrument, weight in current.items()
            )
            if not np.isfinite(portfolio_return):
                portfolio_return = 0.0
            gross = max(1e-12, 1.0 + portfolio_return)
            nav *= gross
            current = {
                instrument: weight * (1.0 + float(day_returns.get(instrument, 0.0) or 0.0)) / gross
                for instrument, weight in current.items()
                if weight > 1e-12
            }

        buy_turnover = 0.0
        sell_turnover = 0.0
        desired = execution_targets.get(date_text(date_value))
        if desired is not None:
            desired = {str(k): float(v) for k, v in desired.items() if float(v) > 1e-12}
            all_instruments = set(current) | set(desired)
            after_sells = dict(current)
            for instrument in all_instruments:
                old = float(after_sells.get(instrument, 0.0))
                target = float(desired.get(instrument, 0.0))
                if target < old:
                    allowed = bool(can_sell.at[date_value, instrument]) if instrument in can_sell.columns else False
                    if allowed:
                        sell_turnover += old - target
                        if target > 1e-12:
                            after_sells[instrument] = target
                        else:
                            after_sells.pop(instrument, None)

            requested_buys = {}
            for instrument, target in desired.items():
                old = float(after_sells.get(instrument, 0.0))
                if target > old:
                    allowed = bool(can_buy.at[date_value, instrument]) if instrument in can_buy.columns else False
                    if allowed:
                        requested_buys[instrument] = target - old
            available_cash = max(0.0, 1.0 - sum(after_sells.values()))
            requested_total = sum(requested_buys.values())
            buy_scale = min(1.0, available_cash / requested_total) if requested_total > 0 else 0.0
            for instrument, increment in requested_buys.items():
                actual = increment * buy_scale
                after_sells[instrument] = after_sells.get(instrument, 0.0) + actual
                buy_turnover += actual
            current = {k: v for k, v in after_sells.items() if v > 1e-12}
            nav *= max(0.0, 1.0 - buy_turnover * BUY_COST - sell_turnover * SELL_COST)

        rows.append({
            "date": date_value,
            "nav": nav,
            "daily_return": nav / start_nav - 1.0 if start_nav > 0 else 0.0,
            "buy_turnover": buy_turnover,
            "sell_turnover": sell_turnover,
            "gross_turnover": buy_turnover + sell_turnover,
            "invested_weight": sum(current.values()),
        })
    return pd.DataFrame(rows).set_index("date")


# =============================================================================
# 12A. 联合 TPE 的独立影子账户与股票级快速模拟
# =============================================================================



def single_cluster_configs(configs: Mapping[str, dict], active_cluster: str) -> Dict[str, dict]:
    """构建等初始资金的影子账户配置；只激活一个策略簇，避免分配金额反向影响其评分。"""
    shadow = deepcopy(dict(configs))
    for cluster_name, config in shadow.items():
        config["capital_weight"] = 1.0 if cluster_name == active_cluster else 0.0
    return shadow


def configs_display_name(cluster_name: str) -> str:
    return str(CLUSTER_CONFIGS[cluster_name]["display_name"])


def calculate_fast_metrics(simulation: pd.DataFrame) -> dict:
    start = max(pd.Timestamp(OPTIMIZATION_START_DATE), simulation.index.min())
    requested_end = min(pd.Timestamp(OPTIMIZATION_END_DATE), simulation.index.max())
    if requested_end <= start:
        requested_end = simulation.index.max()
    sample = simulation.loc[(simulation.index >= start) & (simulation.index <= requested_end)].copy()
    returns = pd.to_numeric(sample["daily_return"], errors="coerce").fillna(0.0)
    n_days = len(returns)
    if n_days < 60:
        raise ValueError("优化评价区间有效交易日不足60天。")
    total_return = float((1.0 + returns).prod() - 1.0)
    annual_return = float((1.0 + total_return) ** (252.0 / n_days) - 1.0)
    volatility = float(returns.std(ddof=1) * math.sqrt(252.0))
    sharpe = float(returns.mean() / returns.std(ddof=1) * math.sqrt(252.0)) if returns.std(ddof=1) > 1e-12 else 0.0
    nav = (1.0 + returns).cumprod()
    drawdown = nav / nav.cummax() - 1.0
    max_drawdown = float(-drawdown.min())
    years = n_days / 252.0
    annual_turnover = float(sample["gross_turnover"].sum() / years) if years > 0 else np.nan
    score = (
        OBJECTIVE_SHARPE_WEIGHT * sharpe
        + OBJECTIVE_ANNUAL_RETURN_WEIGHT * (annual_return / OBJECTIVE_ANNUAL_RETURN_SCALE)
    )
    return {
        "objective": float(score),
        "annual_return": annual_return,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "annual_volatility": volatility,
        "annual_turnover": annual_turnover,
        "annual_turnover_pct": annual_turnover * 100.0,
        "total_return": total_return,
        "evaluation_start": date_text(sample.index.min()),
        "evaluation_end": date_text(sample.index.max()),
        "evaluation_days": int(n_days),
    }


def calculate_fast_metrics_for_period(
    simulation: pd.DataFrame, start_date: str, end_date: str
) -> dict:
    """只用指定期间的净值、换手和成本后日收益计算指标。"""
    start = max(pd.Timestamp(start_date), simulation.index.min())
    end = min(pd.Timestamp(end_date), simulation.index.max())
    sample = simulation.loc[(simulation.index >= start) & (simulation.index <= end)].copy()
    returns = pd.to_numeric(sample["daily_return"], errors="coerce").fillna(0.0)
    n_days = len(returns)
    if n_days < 60:
        raise ValueError(f"评价区间 {start_date} 至 {end_date} 的有效交易日不足 60 天")
    total_return = float((1.0 + returns).prod() - 1.0)
    annual_return = float((1.0 + total_return) ** (252.0 / n_days) - 1.0)
    daily_std = float(returns.std(ddof=1))
    volatility = daily_std * math.sqrt(252.0)
    sharpe = float(returns.mean() / daily_std * math.sqrt(252.0)) if daily_std > 1e-12 else 0.0
    nav = (1.0 + returns).cumprod()
    max_drawdown = float(-(nav / nav.cummax() - 1.0).min())
    annual_turnover = float(sample["gross_turnover"].sum() / (n_days / 252.0))
    return {
        "annual_return": annual_return,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "annual_volatility": volatility,
        "annual_turnover": annual_turnover,
        "annual_turnover_pct": annual_turnover * 100.0,
        "total_return": total_return,
        "evaluation_start": date_text(sample.index.min()),
        "evaluation_end": date_text(sample.index.max()),
        "evaluation_days": int(n_days),
    }


def build_trend_panels_from_cache(
    configs: Mapping[str, dict], cache: Mapping[str, object]
) -> Dict[str, pd.DataFrame]:
    panels = {}
    trade_index = pd.DatetimeIndex(cache["trade_dates"])
    for cluster_name, config in configs.items():
        if not config.get("use_defensive_compensation", False):
            continue
        full_close = cache["trend_close"][config["trend_index"]].sort_index()
        window = int(config["trend_ma_window"])
        full_ma = full_close.rolling(window, min_periods=window).mean()
        close = full_close.reindex(trade_index).ffill()
        ma = full_ma.reindex(trade_index).ffill()
        panels[cluster_name] = pd.DataFrame({
            "date": trade_index,
            "close": close.to_numpy(),
            "ma": ma.to_numpy(),
            "risk_on": (close >= ma).where(ma.notna(), True).to_numpy(dtype=bool),
            "trend_index_requested": config["trend_index"],
            "trend_index_actual": config["trend_index"],
        })
    return panels


def prepare_strategy_data_from_optimization_cache(
    configs: Mapping[str, dict], cache: Mapping[str, object]
) -> None:
    """用已查询的数据为最佳配置建立BigTrader运行时对象，不重复访问DAI。"""
    global _RUNTIME_DATA
    progress("[最佳配置1/4] 使用缓存生成四个策略簇的最终选股信号")
    schedules, next_date_map = build_schedules(cache["trade_dates"], configs)
    schedules, factor_availability_audit = delay_schedules_until_factor_ready(
        cache["base_panel"], cache["factor_panels"], schedules, configs
    )
    cluster_targets, selection_audit, score_audit = build_cluster_signals(
        cache["base_panel"], cache["factor_panels"], schedules, configs
    )
    progress("[最佳配置2/4] 使用缓存生成趋势覆盖层")
    trend_panels = build_trend_panels_from_cache(configs, cache)
    trend_lookup = {
        name: {date_text(row["date"]): row.to_dict() for _, row in panel.iterrows()}
        for name, panel in trend_panels.items()
    }
    expected_trade_dates = {}
    for schedule in schedules.values():
        expected_trade_dates.update(schedule["expected_trade_dates"])
    _RUNTIME_DATA = {
        "trade_dates": cache["trade_dates"],
        "schedules": schedules,
        "next_date_map": next_date_map,
        "expected_trade_dates": expected_trade_dates,
        "cluster_targets": cluster_targets,
        "trend_panels": trend_panels,
        "trend_lookup": trend_lookup,
    }
    preserved = {
        key: RESEARCH_ARTIFACTS[key]
        for key in (
            "optimization_study", "optimization_results", "optimization_top_results",
            "optimization_output_paths", "best_config",
            "walk_forward_study", "walk_forward_results", "walk_forward_fold_results",
            "walk_forward_output_paths", "factor_diagnostics", "factor_diagnostic_series",
            "preprocessing_comparison", "factor_diagnostic_output_paths", "factor_ic_chart_path",
            "ablation_results", "ablation_output_path",
        )
        if key in RESEARCH_ARTIFACTS
    }
    RESEARCH_ARTIFACTS.clear()
    RESEARCH_ARTIFACTS.update({
        "configs": deepcopy(configs),
        "factor_source_audit": cache["factor_source_audit"],
        "factor_panel_summary": cache["factor_panel_summary"].copy(),
        "factor_availability_audit": factor_availability_audit,
        "factor_warmup_audit": cache["factor_warmup_audit"].copy(),
        "factor_query_start": cache["factor_query_start"],
        "selection_audit": selection_audit,
        "score_audit": score_audit,
        "cluster_targets": cluster_targets,
        "trend_panels": trend_panels,
        "capital_usage_records": [],
        "execution_audit": [],
        "order_event_audit": [],
        "trade_event_audit": [],
        "cluster_trade_allocation_records": [],
        "cluster_ledger_daily_records": [],
        "cluster_position_records": [],
        "cluster_reconciliation_records": [],
        "unmatched_trade_records": [],
        **preserved,
    })
    progress("[最佳配置3/4] BigTrader运行时索引建立完成")
    gc.collect()
    progress("[最佳配置4/4] 准备运行唯一一次原生回测")


# =============================================================================
# 13. 数据准备与运行入口
# =============================================================================

def prepare_strategy_data(configs: Mapping[str, dict]) -> None:
    global _RUNTIME_DATA
    progress("[1/7] 获取交易日历并建立四套独立调仓日程")
    trade_dates = query_trading_calendar()
    schedules, next_date_map = build_schedules(trade_dates, configs)
    all_signal_dates = sorted(
        {date for schedule in schedules.values() for date in schedule["signal_dates"]}
    )
    progress(
        "调仓截面数量：" + "；".join(
            f"{configs[name]['display_name']}={len(schedule['signal_dates'])}"
            for name, schedule in schedules.items()
        )
    )

    progress("[2/7] 查询所有策略簇共用的点时股票池截面")
    base_panel = query_base_panel(all_signal_dates)
    query_start = query_factor_warmup_start()

    progress("[3/7] 按原始代码公式从基础字段构建七个因子（不查询同名成品因子）")
    factor_panels, source_audit = build_raw_factor_panels(base_panel, schedules, query_start)
    factor_panel_summary = summarize_factor_panels(factor_panels)
    for row in factor_panel_summary.itertuples(index=False):
        progress(
            f"因子覆盖：{row.factor}，{row.start_date}至{row.end_date}，"
            f"{row.cross_sections}个截面、{row.rows:,}行"
        )
    schedules, factor_availability_audit = delay_schedules_until_factor_ready(
        base_panel, factor_panels, schedules, configs
    )
    warmup_audit = validate_initial_factor_warmup(
        base_panel, factor_panels, schedules, configs
    )

    progress("[4/7] 逐策略簇完成方向统一、组内排名线性合成与市值组选股")
    cluster_targets, selection_audit, score_audit = build_cluster_signals(
        base_panel, factor_panels, schedules, configs
    )

    progress("[5/7] 构建中证1000MA60与沪深300MA60趋势覆盖层")
    trend_panels = build_all_trends(configs, trade_dates)
    trend_lookup = {
        cluster_name: {
            date_text(row["date"]): row.to_dict()
            for _, row in panel.iterrows()
        }
        for cluster_name, panel in trend_panels.items()
    }

    expected_trade_dates = {}
    for schedule in schedules.values():
        expected_trade_dates.update(schedule["expected_trade_dates"])

    progress("[6/7] 建立回测运行时索引与审计对象")
    _RUNTIME_DATA = {
        "trade_dates": trade_dates,
        "schedules": schedules,
        "next_date_map": next_date_map,
        "expected_trade_dates": expected_trade_dates,
        "cluster_targets": cluster_targets,
        "trend_panels": trend_panels,
        "trend_lookup": trend_lookup,
    }
    RESEARCH_ARTIFACTS.clear()
    RESEARCH_ARTIFACTS.update(
        {
            "configs": deepcopy(configs),
            "factor_source_audit": source_audit,
            "factor_panel_summary": factor_panel_summary,
            "factor_availability_audit": factor_availability_audit,
            "factor_warmup_audit": warmup_audit,
            "factor_query_start": query_start,
            "selection_audit": selection_audit,
            "score_audit": score_audit,
            "cluster_targets": cluster_targets,
            "trend_panels": trend_panels,
            "capital_usage_records": [],
            "execution_audit": [],
            "order_event_audit": [],
            "trade_event_audit": [],
            "cluster_trade_allocation_records": [],
            "cluster_ledger_daily_records": [],
            "cluster_position_records": [],
            "cluster_reconciliation_records": [],
            "unmatched_trade_records": [],
        }
    )
    if KEEP_FACTOR_PANELS_IN_MEMORY:
        RESEARCH_ARTIFACTS["factor_panels"] = factor_panels
    del base_panel
    gc.collect()
    progress("[7/7] 数据准备完成")


def performance_summary_frame(performance) -> pd.DataFrame:
    summary = getattr(performance, "summary", performance)
    if isinstance(summary, pd.DataFrame):
        return summary.copy()
    if isinstance(summary, pd.Series):
        return summary.rename("数值").reset_index().rename(columns={"index": "指标"})
    if isinstance(summary, Mapping):
        return pd.DataFrame([dict(summary)])
    return pd.DataFrame({"原生回测摘要": [str(summary)]})


def static_anchor_weights(configs: Mapping[str, dict]) -> np.ndarray:
    """返回当前 TPE trial 的四簇静态资金锚，并校验软硬边界与 100% 合计。"""
    values = np.asarray([float(configs[name]["capital_weight"]) for name in configs], dtype=float)
    if len(values) != 4 or not np.isfinite(values).all() or np.any(values < MIN_CLUSTER_CAPITAL_WEIGHT):
        raise ValueError("静态资金锚必须包含四个满足最低边界的有限权重。")
    if values[0] > MAX_CLUSTER_1_CAPITAL_WEIGHT or not np.isclose(values.sum(), 1.0, atol=1e-10):
        raise ValueError("静态资金锚必须满足上限约束且权重之和为100%。")
    return values.copy()


def band_rebalance_deviation(weights: Sequence[float], anchor: Sequence[float]) -> Tuple[float, np.ndarray]:
    actual = np.asarray(weights, dtype=float)
    target = np.asarray(anchor, dtype=float)
    total = float(0.5 * np.abs(actual - target).sum())
    relative = np.divide(
        np.abs(actual - target), target,
        out=np.zeros_like(actual), where=target > 1e-12,
    )
    return total, relative


def plot_band_native_account_positions(ledger_daily: pd.DataFrame) -> Optional[str]:
    """按成交级账本绘制四个账户的持仓市值、现金、净值和目标资金，避免使用旧 ownership_shares。"""
    if ledger_daily.empty:
        return None
    configure_chinese_font()
    value_column = next((c for c in ledger_daily.columns if "实际持仓市值" in str(c)), None)
    cash_column = next((c for c in ledger_daily.columns if "实际现金" in str(c)), None)
    nav_column = next((c for c in ledger_daily.columns if "账本净值" in str(c)), None)
    target_column = next((c for c in ledger_daily.columns if "目标资金" in str(c)), None)
    if not all((value_column, cash_column, nav_column, target_column)):
        warnings.warn("策略簇账本字段不完整，跳过各账户仓位图。")
        return None
    figure, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
    for axis, cluster_name in zip(np.atleast_1d(axes), _ACTIVE_CONFIGS):
        frame = ledger_daily[ledger_daily["cluster"] == cluster_name].copy()
        frame["date"] = pd.to_datetime(frame["date"])
        frame = frame.sort_values("date")
        scale = 10_000.0
        axis.stackplot(
            frame["date"],
            pd.to_numeric(frame[value_column], errors="coerce").fillna(0.0) / scale,
            pd.to_numeric(frame[cash_column], errors="coerce").fillna(0.0) / scale,
            labels=["实际持仓市值", "实际现金"], colors=["#5B8FD1", "#C9C9C9"], alpha=0.82,
        )
        axis.plot(frame["date"], pd.to_numeric(frame[nav_column], errors="coerce") / scale, color="#C44E52", linewidth=1.2, label="账本净值")
        axis.plot(frame["date"], pd.to_numeric(frame[target_column], errors="coerce") / scale, color="#69A85B", linewidth=1.0, linestyle="--", label="目标资金")
        axis.set_title(configs_display_name(cluster_name))
        axis.set_ylabel("资金（万元）")
        axis.grid(alpha=0.25)
        axis.legend(loc="upper left", ncol=4)
    axes[-1].set_xlabel("日期")
    figure.suptitle("四个策略簇的成交级账户仓位", y=1.002)
    figure.tight_layout()
    path = os.path.join(result_output_directory(), "四策略簇账户仓位图.png")
    figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    return path


def register_pending_cluster_trades_subset(
    context, previous_components: Mapping[str, dict], components: Mapping[str, dict], active_clusters: Sequence[str]
) -> None:
    previous = component_weights_by_cluster(previous_components)
    current = component_weights_by_cluster(components)
    pending = {}
    for cluster_name in active_clusters:
        instruments = set(previous.get(cluster_name, {})) | set(current.get(cluster_name, {}))
        for instrument in instruments:
            delta = float(current.get(cluster_name, {}).get(instrument, 0.0)) - float(previous.get(cluster_name, {}).get(instrument, 0.0))
            if abs(delta) > 1e-12:
                pending.setdefault(str(instrument), []).append({"cluster": cluster_name, "weight_delta": delta})
    context.pending_cluster_trade_requests = pending


def component_instruments_for_clusters(components: Mapping[str, dict], clusters: Sequence[str]) -> set:
    values = set()
    for cluster_name in clusters:
        sleeve = components.get(cluster_name, {})
        for part in ("factor", "defensive"):
            values.update(map(str, sleeve.get(part, {}).keys()))
    return values


# v2 与旧部署隔离：旧版本不会在部署首日建仓，也不会保留当日到下一开盘的日历映射。
SIMULATION_STATE_PREFIX = "four_cluster_fixed_best_v2"


def initial_simulation_signal_required(context) -> bool:
    """仅在模拟盘首次部署（或人工清空 v2 状态）时触发一次初始建仓信号。"""
    if not TRADING_DATE:
        return False
    store = getattr(context, "user_store", None)
    return store is not None and not bool(
        store.get(f"{SIMULATION_STATE_PREFIX}:initial_signal_submitted", False)
    )


def mark_initial_simulation_signal_submitted(context, signal_date: str) -> None:
    """记录初始建仓指令已成功发出，避免次日任务重复建仓。"""
    store = getattr(context, "user_store", None)
    if store is None:
        return
    store[f"{SIMULATION_STATE_PREFIX}:initial_signal_submitted"] = True
    store[f"{SIMULATION_STATE_PREFIX}:initial_signal_date"] = str(signal_date)


def next_execution_date(context, today: str) -> str:
    """返回信号日在下一交易日开盘的执行日期，不读取未来行情数据。"""
    known_date = _RUNTIME_DATA["next_date_map"].get(today, "")
    if known_date:
        return str(known_date)
    if not TRADING_DATE:
        return ""
    try:
        # BigTrader 日频模拟盘提供交易日历接口；这只查询日历，不会引入未来价格或因子。
        return pd.Timestamp(context.add_trading_days(today, 1)).strftime("%Y-%m-%d")
    except Exception as exc:
        context.logger.warning(f"{today} 无法取得下一交易日：{exc}")
        return ""


def restore_simulation_state(context, today: str) -> None:
    """为日频模拟交易恢复上一任务保存的簇内目标与周期执行日。

    回测通常是连续进程，而模拟交易任务可能每天重启。若不恢复状态，未在当日调仓的策略簇
    会被误认为没有目标股票；周期再平衡也会从零重新计数。首次运行没有历史状态时，使用
    当前日期之前最近一次已生成的簇内信号重建目标，后续运行使用 context.user_store。
    """
    if getattr(context, "simulation_state_restored", False):
        return
    store = getattr(context, "user_store", None)
    stored_targets = store.get(f"{SIMULATION_STATE_PREFIX}:targets") if store is not None else None
    if isinstance(stored_targets, Mapping) and set(stored_targets) == set(_ACTIVE_CONFIGS):
        context.current_cluster_targets = {
            name: list(stored_targets.get(name, [])) for name in _ACTIVE_CONFIGS
        }
    else:
        # 首次部署或清空状态后的安全回退：取每簇最近一个不晚于 today 的已知信号。
        for cluster_name in _ACTIVE_CONFIGS:
            schedule = context.cluster_targets_by_signal_date.get(cluster_name, {})
            available = [date_value for date_value in schedule if date_value <= today]
            if available:
                context.current_cluster_targets[cluster_name] = list(schedule[max(available)])

    stored_nav = store.get(f"{SIMULATION_STATE_PREFIX}:sleeve_nav") if store is not None else None
    if isinstance(stored_nav, Mapping) and set(stored_nav) == set(_ACTIVE_CONFIGS):
        context.latest_sleeve_nav = {name: float(stored_nav[name]) for name in _ACTIVE_CONFIGS}
        total = sum(context.latest_sleeve_nav.values())
        if total > 1e-12:
            context.latest_sleeve_weights = {
                name: value / total for name, value in context.latest_sleeve_nav.items()
            }
    stored_risk = store.get(f"{SIMULATION_STATE_PREFIX}:risk_on") if store is not None else None
    for cluster_name, config in _ACTIVE_CONFIGS.items():
        if not config.get("use_defensive_compensation", False):
            continue
        # 首次运行以今日状态为基准，避免每日任务重启时把“None -> 当前状态”误判为趋势切换。
        fallback = trend_state_for_date(cluster_name, today)
        context.previous_risk_on[cluster_name] = bool(
            stored_risk.get(cluster_name, fallback)
        ) if isinstance(stored_risk, Mapping) else bool(fallback)
    context.simulation_state_restored = True


def persist_simulation_state(
    context,
    execution_date: Optional[str] = None,
    signal_date: Optional[str] = None,
) -> None:
    """将下一次模拟运行需要的最小状态写入 BigTrader 的持久化用户存储。"""
    store = getattr(context, "user_store", None)
    if store is None:
        return
    store[f"{SIMULATION_STATE_PREFIX}:targets"] = {
        name: list(context.current_cluster_targets.get(name, [])) for name in _ACTIVE_CONFIGS
    }
    store[f"{SIMULATION_STATE_PREFIX}:sleeve_nav"] = {
        name: float(context.latest_sleeve_nav.get(name, 0.0)) for name in _ACTIVE_CONFIGS
    }
    store[f"{SIMULATION_STATE_PREFIX}:risk_on"] = {
        name: bool(context.previous_risk_on.get(name, True)) for name in _ACTIVE_CONFIGS
    }
    if execution_date:
        store[f"{SIMULATION_STATE_PREFIX}:last_periodic_execution_date"] = str(execution_date)
    if signal_date:
        # 日频模拟任务会跨日重启；下一开盘的成交可能由平台在本次 Python 任务结束后处理。
        # 在成交回调可用前，以已发出的信号日作为周期锚，防止每天重复提交同一笔周期再平衡。
        store[f"{SIMULATION_STATE_PREFIX}:last_periodic_signal_date"] = str(signal_date)


def initialize_band_rebalance(context: "bigtrader.IContext"):
    initialize(context)
    anchor = _RUNTIME_DATA["band_rebalance_anchor_weights"]
    context.band_anchor_weights = {name: float(value) for name, value in zip(_ACTIVE_CONFIGS, anchor)}
    context.active_cluster_capital_weights = dict(context.band_anchor_weights)
    context.latest_sleeve_nav = {name: CAPITAL_BASE * value for name, value in context.band_anchor_weights.items()}
    context.latest_sleeve_weights = dict(context.band_anchor_weights)
    context.band_last_rebalance_index = 0
    context.band_date_index = {date_text(value): index for index, value in enumerate(_RUNTIME_DATA["trade_dates"])}
    store = getattr(context, "user_store", None)
    persisted_date = (
        store.get(f"{SIMULATION_STATE_PREFIX}:last_periodic_execution_date")
        if store is not None else None
    )
    if str(persisted_date) not in context.band_date_index and store is not None:
        persisted_date = store.get(f"{SIMULATION_STATE_PREFIX}:last_periodic_signal_date")
    if str(persisted_date) in context.band_date_index:
        context.band_last_rebalance_index = context.band_date_index[str(persisted_date)]
    context.band_rebalance_records = RESEARCH_ARTIFACTS["band_rebalance_runtime_records"]
    context.band_state_records = RESEARCH_ARTIFACTS["band_rebalance_state_records"]
    # 原生订单在下一交易日才撮合。未完成收盘对账前，不能把同一个偏离连续当作新触发。
    context.band_pending_execution_date = None
    context.band_pending_record = None
    context.logger.info("周期再平衡器已初始化：簇内调仓与簇间静态资金锚再平衡分离。")


def handle_data_band_rebalance(context: "bigtrader.IContext", data: "bigtrader.IBarData"):
    """簇内交易照常执行；仅在固定周期到达时将四簇整体拉回静态资金锚。"""
    today = pd.Timestamp(data.current_dt).strftime("%Y-%m-%d")
    restore_simulation_state(context, today)
    initial_deployment_requested = initial_simulation_signal_required(context)
    changed_clusters, trend_changed_clusters = [], []
    for cluster_name, config in _ACTIVE_CONFIGS.items():
        new_targets = context.cluster_targets_by_signal_date[cluster_name].get(today)
        if new_targets is not None:
            context.current_cluster_targets[cluster_name] = list(new_targets)
            changed_clusters.append(cluster_name)
        if config.get("use_defensive_compensation", False):
            risk_on = trend_state_for_date(cluster_name, today)
            old_state = context.previous_risk_on.get(cluster_name)
            if old_state is None or bool(old_state) != bool(risk_on):
                trend_changed_clusters.append(cluster_name)
            context.previous_risk_on[cluster_name] = bool(risk_on)

    expected_trade_date = _RUNTIME_DATA["expected_trade_dates"].get(
        today, next_execution_date(context, today)
    )
    rebalance_triggered, trigger_detail = band_runtime_trigger(context, today)
    initial_deployment = bool(initial_deployment_requested and expected_trade_date)
    if initial_deployment:
        # 首次部署即用截至今日最近一个有效截面建仓；信号在今日收盘后提交，
        # 保持 VMatchAt.NEXT_BAR_OPEN 所规定的下一交易日开盘撮合时序。
        rebalance_triggered = True
        trigger_detail["触发原因"] = "首次部署建仓"
    elif initial_deployment_requested:
        context.logger.warning(
            f"{today} 无法定位下一交易日，首次部署建仓信号暂不提交。"
        )
    trigger_detail["是否实际触发"] = bool(rebalance_triggered)
    trigger_detail["是否首次部署建仓"] = bool(initial_deployment)
    trigger_detail["簇内目标变动簇数"] = len(changed_clusters)
    context.band_state_records.append(deepcopy(trigger_detail))
    if not changed_clusters and not trend_changed_clusters and not rebalance_triggered:
        return
    if rebalance_triggered:
        context.active_cluster_capital_weights = dict(context.band_anchor_weights)
        affected_clusters = list(_ACTIVE_CONFIGS)
        context.band_pending_execution_date = expected_trade_date or None
    else:
        # 非周期日按已对账的实际簇净值维持簇间比例，避免簇内调仓重置其他三簇。
        context.active_cluster_capital_weights = dict(context.latest_sleeve_weights)
        affected_clusters = list(dict.fromkeys(changed_clusters + trend_changed_clusters))

    components = build_sleeve_component_weights(context, today)
    aggregate_targets = aggregate_component_weights(components)
    previous_components = context.last_sleeve_components
    impacted = component_instruments_for_clusters(previous_components, affected_clusters)
    impacted.update(component_instruments_for_clusters(components, affected_clusters))
    positions = current_positions(context)
    if rebalance_triggered:
        impacted.update(str(instrument) for instrument, position in positions.items() if position_amount(position) > 0)
    register_pending_cluster_trades_subset(context, previous_components, components, affected_clusters)

    submitted = rejected = 0
    errors = {}
    for instrument in sorted(impacted):
        target_weight = float(aggregate_targets.get(instrument, 0.0))
        success, _, error = submit_target_percent(context, instrument, target_weight)
        submitted += int(success)
        rejected += int(not success)
        if error:
            errors[error] = errors.get(error, 0) + 1
    context.last_sleeve_components = components
    context.last_aggregate_targets = aggregate_targets
    audit = {
        "signal_date": today,
        "expected_trade_date": expected_trade_date,
        "initial_deployment": bool(initial_deployment),
        "factor_rebalance_clusters": list(changed_clusters),
        "trend_change_clusters": list(trend_changed_clusters),
        "band_rebalance_triggered": bool(rebalance_triggered),
        "band_affected_clusters": affected_clusters,
        "aggregate_target_count": len(aggregate_targets),
        "submitted_order_count": submitted,
        "immediate_rejected_order_count": rejected,
        "immediate_errors": errors,
    }
    context.execution_audit.append(audit)
    # 当日新信号已写入 current_cluster_targets；持久化后即使次日任务重启也不会丢失。
    persist_simulation_state(context, signal_date=today if rebalance_triggered else None)
    if initial_deployment and submitted > 0:
        mark_initial_simulation_signal_submitted(context, today)
    if rebalance_triggered:
        trigger_detail.update({
            "执行日期": expected_trade_date,
            "影响簇数": len(affected_clusters),
            "提交订单数": submitted,
            "即时拒单数": rejected,
            "执行后是否完成对账": False,
        })
        context.band_rebalance_records.append(trigger_detail)
        context.band_pending_record = trigger_detail
    if VERBOSE_REBALANCE or rebalance_triggered or rejected > 0:
        context.logger.info(
            f"{today} 周期再平衡={rebalance_triggered} | 首次部署={initial_deployment} | 簇内变化={changed_clusters} | "
            f"影响订单={len(impacted)} | 提交={submitted} | 拒单={rejected}"
        )


def after_trading_band_rebalance(context: "bigtrader.IContext", data: "bigtrader.IBarData"):
    """收盘后先完成成交级对账，再重置周期计数的基准执行日。"""
    after_trading(context, data)
    today = pd.Timestamp(data.current_dt).strftime("%Y-%m-%d")
    # 每日收盘后保存最新簇净值；非周期日也必须保存，供下一次模拟任务恢复资金比例。
    persist_simulation_state(context)
    pending_date = getattr(context, "band_pending_execution_date", None)
    if not pending_date or str(today) < str(pending_date):
        return

    # 下一交易日撮合后的账本已在 after_trading 中更新；从此刻起重新计算周期交易日间隔。
    context.band_last_rebalance_index = int(
        context.band_date_index.get(today, context.band_last_rebalance_index)
    )
    _, actual, total_deviation, _, _, _ = band_runtime_deviation(context)
    record = getattr(context, "band_pending_record", None)
    if isinstance(record, dict):
        record.update({
            "实际对账日期": today,
            "执行后是否完成对账": True,
            "执行后总交易偏离": float(total_deviation),
        })
        for name, value in zip(_ACTIVE_CONFIGS, actual):
            record[f"{configs_display_name(name)}_执行后实际权重"] = float(value)
    context.band_pending_execution_date = None
    context.band_pending_record = None
    persist_simulation_state(context, execution_date=today)


def band_trigger_consistency_audit(fast_decisions: pd.DataFrame, native_decisions: pd.DataFrame) -> pd.DataFrame:
    """逐笔核对快速模拟与原生回测的周期决策日、执行日，不能只比较触发次数。"""
    columns = ["决策日期", "执行日期"]

    def normalized(frame: pd.DataFrame, source: str) -> pd.DataFrame:
        result = frame.reindex(columns=columns).copy()
        for column in columns:
            result[column] = pd.to_datetime(result[column], errors="coerce").dt.strftime("%Y-%m-%d")
        result = result.dropna(subset=columns).drop_duplicates().sort_values(columns).reset_index(drop=True)
        result[f"{source}是否触发"] = True
        return result

    fast = normalized(fast_decisions, "快速模拟")
    native = normalized(native_decisions, "原生回测")
    audit = fast.merge(native, on=columns, how="outer").sort_values(columns).reset_index(drop=True)
    audit["快速模拟是否触发"] = audit["快速模拟是否触发"].fillna(False).astype(bool)
    audit["原生回测是否触发"] = audit["原生回测是否触发"].fillna(False).astype(bool)
    audit["是否一致"] = audit["快速模拟是否触发"] == audit["原生回测是否触发"]
    audit["差异说明"] = np.where(
        audit["是否一致"], "一致",
        np.where(audit["快速模拟是否触发"], "原生缺少该周期触发", "原生额外周期触发"),
    )
    if not bool(audit["是否一致"].all()):
        warnings.warn("快速模拟与原生回测的周期触发日期不一致，请检查原生周期再平衡日度状态审计表。")
    return audit


def run_prepared_bigtrader_band_rebalance():
    progress("开始BigTrader原生回测：区间触发簇间再平衡")
    performance = bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date=START_DATE,
        end_date=END_DATE,
        capital_base=CAPITAL_BASE,
        benchmark=BENCHMARK,
        initialize=initialize_band_rebalance,
        handle_data=handle_data_band_rebalance,
        after_trading=after_trading_band_rebalance,
        handle_order=handle_order,
        handle_trade=handle_trade,
        order_price_field_buy="open",
        order_price_field_sell="open",
        volume_limit=VOLUME_LIMIT,
    )
    progress("BigTrader原生回测完成")
    finalize_artifacts(performance)
    try:
        display(performance.summary)
    except Exception:
        display(performance)
    return performance


def band_parameter_display_row(params: Mapping[str, object]) -> dict:
    """周期再平衡只展示周期参数，不存在任何偏离阈值参数。"""
    return dict(params)


def band_runtime_deviation(context) -> Tuple[np.ndarray, np.ndarray, float, np.ndarray, np.ndarray, bool]:
    """总偏离仅用于审计展示，不参与任何再平衡判断或参数搜索。"""
    anchor = np.asarray([context.band_anchor_weights[name] for name in _ACTIVE_CONFIGS], dtype=float)
    sleeve_nav = np.asarray([context.latest_sleeve_nav.get(name, 0.0) for name in _ACTIVE_CONFIGS], dtype=float)
    total_nav = float(sleeve_nav.sum())
    actual = sleeve_nav / total_nav if total_nav > 1e-12 else anchor.copy()
    total_deviation, relative = band_rebalance_deviation(actual, anchor)
    return anchor, actual, total_deviation, relative, np.full_like(anchor, np.nan), True


def band_runtime_trigger(context, today: str) -> Tuple[bool, dict]:
    """唯一触发条件：距离上次完成再平衡达到候选周期。"""
    anchor, actual, total_deviation, _, _, _ = band_runtime_deviation(context)
    interval = int(_RUNTIME_DATA["band_rebalance_params"]["周期再平衡间隔"])
    elapsed = int(context.band_date_index.get(today, 0) - context.band_last_rebalance_index)
    waiting_reconcile = bool(getattr(context, "band_pending_execution_date", None))
    execution_date = next_execution_date(context, today)
    # 末个回测交易日没有下一交易日开盘，不能提交无法按既定时序成交的周期再平衡订单。
    triggered = bool(not waiting_reconcile and bool(execution_date) and elapsed >= interval)
    detail = {
        "决策日期": today,
        "距上次执行交易日": elapsed,
        "周期再平衡间隔": interval,
        "决策日总交易偏离（仅审计）": float(total_deviation),
        "触发原因": "周期再平衡" if triggered else "",
        "是否等候执行对账": waiting_reconcile,
        "下一交易日开盘可执行": bool(execution_date),
    }
    for name, actual_weight, anchor_weight in zip(_ACTIVE_CONFIGS, actual, anchor):
        detail[f"{configs_display_name(name)}_决策日实际权重"] = float(actual_weight)
        detail[f"{configs_display_name(name)}_相对静态锚偏离（仅审计）"] = float(abs(actual_weight - anchor_weight))
    return triggered, detail


def build_joint_cluster_shadow_inputs(
    configs: Mapping[str, dict], cache: Mapping[str, object]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """分别模拟四簇的单位资金账户，保留日收益和簇内股票换手。"""
    returns, turnovers = {}, {}
    for number, cluster_name in enumerate(configs, start=1):
        progress(
            f"[联合搜索] 构建候选簇影子账户 {number}/4："
            f"{configs[cluster_name]['display_name']}"
        )
        simulation = fast_portfolio_simulation(
            single_cluster_configs(configs, cluster_name), cache
        )
        returns[cluster_name] = pd.to_numeric(
            simulation["daily_return"], errors="coerce"
        ).fillna(0.0)
        turnovers[cluster_name] = pd.to_numeric(
            simulation["gross_turnover"], errors="coerce"
        ).fillna(0.0)
    index = pd.DatetimeIndex(cache["trade_dates"])
    return (
        pd.DataFrame(returns).reindex(index).fillna(0.0),
        pd.DataFrame(turnovers).reindex(index).fillna(0.0),
    )


def simulate_joint_periodic_portfolio(
    shadow_returns: pd.DataFrame,
    shadow_turnovers: pd.DataFrame,
    anchor: Sequence[float],
    periodic_days: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """在簇内成本之后，按固定交易日间隔执行含跨簇成本的静态锚再平衡。

    时间顺序：决策日收盘后发现距上次实际执行日达到 periodic_days，记录待执行指令；
    下一交易日开盘先按当前簇净值卖出超配簇、买入低配簇，分别扣 SELL_COST 和 BUY_COST，
    再把四簇净值调回 anchor。最后一个交易日没有下一开盘，因此不会生成待执行指令。
    """
    ordered = list(shadow_returns.columns)
    returns = shadow_returns.reindex(columns=ordered).fillna(0.0)
    turnovers = shadow_turnovers.reindex(index=returns.index, columns=ordered).fillna(0.0)
    anchor_array = np.asarray(anchor, dtype=float)
    anchor_array = anchor_array / anchor_array.sum()
    values = anchor_array.copy()
    pending = None
    last_execution_index = 0
    rows, decisions = [], []

    for index, (date_value, return_row) in enumerate(returns.iterrows()):
        cross_turnover = 0.0
        cross_cost = 0.0
        execution_reason = ""
        if pending is not None:
            before_total = float(values.sum())
            target_values = anchor_array * before_total
            sell_fraction = float(np.maximum(values - target_values, 0.0).sum() / before_total) if before_total > 0 else 0.0
            buy_fraction = float(np.maximum(target_values - values, 0.0).sum() / before_total) if before_total > 0 else 0.0
            cross_turnover = sell_fraction + buy_fraction
            cross_cost = sell_fraction * SELL_COST + buy_fraction * BUY_COST
            values = anchor_array * before_total * max(0.0, 1.0 - cross_cost)
            last_execution_index = index
            execution_reason = "周期再平衡"
            pending.update({
                "执行日前总资产": before_total,
                "执行日跨簇总换手": cross_turnover,
                "执行日跨簇成本率": cross_cost,
            })
            decisions.append(pending)
            pending = None

        start_nav = float(values.sum())
        start_weights = values / start_nav if start_nav > 1e-12 else anchor_array.copy()
        internal_turnover = float(np.dot(
            start_weights,
            pd.to_numeric(turnovers.loc[date_value], errors="coerce").fillna(0.0).to_numpy(dtype=float),
        ))
        day_return = pd.to_numeric(return_row, errors="coerce").fillna(0.0).to_numpy(dtype=float)
        values *= np.maximum(1e-12, 1.0 + day_return)
        end_nav = float(values.sum())
        actual_weights = values / end_nav if end_nav > 1e-12 else anchor_array.copy()
        total_deviation = float(0.5 * np.abs(actual_weights - anchor_array).sum())
        elapsed = int(index - last_execution_index)

        if pending is None and index < len(returns) - 1 and elapsed >= int(periodic_days):
            pending = {
                "决策日期": date_text(date_value),
                "执行日期": date_text(returns.index[index + 1]),
                "触发原因": "周期再平衡",
                "距上次执行交易日": elapsed,
                "决策日总交易偏离": total_deviation,
                "周期再平衡间隔": int(periodic_days),
            }

        row = {
            "date": pd.Timestamp(date_value),
            "nav": end_nav,
            "daily_return": end_nav / start_nav - 1.0 if start_nav > 1e-12 else 0.0,
            "buy_turnover": internal_turnover / 2.0 + cross_turnover / 2.0,
            "sell_turnover": internal_turnover / 2.0 + cross_turnover / 2.0,
            "gross_turnover": internal_turnover + cross_turnover,
            "簇内股票换手": internal_turnover,
            "跨簇资金换手": cross_turnover,
            "跨簇资金成本率": cross_cost,
            "总交易偏离": total_deviation,
        }
        for name, value, weight in zip(ordered, values, actual_weights):
            row[f"{name}_影子净值"] = float(value)
            row[f"{name}_实际权重"] = float(weight)
        rows.append(row)
    return pd.DataFrame(rows).set_index("date"), pd.DataFrame(decisions)


def calculate_joint_walk_forward_metrics(simulation: pd.DataFrame) -> Tuple[dict, pd.DataFrame]:
    """仅使用 2022–2024 三段验证折评分，并对全股票层面总换手进行惩罚。"""
    rows = []
    for fold_name, train_start, train_end, valid_start, valid_end in WALK_FORWARD_FOLDS:
        metrics = calculate_fast_metrics_for_period(simulation, valid_start, valid_end)
        passed = (
            metrics["max_drawdown"] <= MAX_VALIDATION_DRAWDOWN
            and metrics["annual_turnover"] <= MAX_VALIDATION_ANNUAL_TURNOVER
        )
        rows.append({
            "验证折": fold_name,
            "训练开始": train_start,
            "训练结束": train_end,
            "验证开始": valid_start,
            "验证结束": valid_end,
            "是否通过约束": bool(passed),
            **metrics,
        })
    detail = pd.DataFrame(rows)
    mean_sharpe = float(detail["sharpe"].mean())
    mean_return = float(detail["annual_return"].mean())
    worst_drawdown = float(detail["max_drawdown"].max())
    mean_turnover = float(detail["annual_turnover"].mean())
    sharpe_std = float(detail["sharpe"].std(ddof=0))
    passed = bool(detail["是否通过约束"].all())
    objective = (
        JOINT_SCORE_SHARPE_WEIGHT * mean_sharpe
        + JOINT_SCORE_RETURN_WEIGHT * mean_return / OBJECTIVE_ANNUAL_RETURN_SCALE
        - JOINT_SCORE_DRAWDOWN_PENALTY * worst_drawdown / MAX_VALIDATION_DRAWDOWN
        - JOINT_SCORE_TURNOVER_PENALTY * mean_turnover / MAX_VALIDATION_ANNUAL_TURNOVER
        - JOINT_SCORE_STABILITY_PENALTY * sharpe_std
    )
    if not passed:
        objective = -10.0 - worst_drawdown - mean_turnover / MAX_VALIDATION_ANNUAL_TURNOVER
    return {
        "objective": float(objective),
        "walk_forward_passed": passed,
        "mean_validation_annual_return": mean_return,
        "mean_validation_sharpe": mean_sharpe,
        "validation_sharpe_std": sharpe_std,
        "worst_validation_drawdown": worst_drawdown,
        "mean_validation_annual_turnover": mean_turnover,
        "validation_fold_count": int(len(detail)),
    }, detail


def joint_periodic_results_frame(study, records: Mapping[int, dict]) -> pd.DataFrame:
    rows = []
    for number, record in records.items():
        config = record["config"]
        metrics = record["metrics"]
        row = {
            "试验编号": int(number),
            "周期再平衡间隔": int(record["periodic_days"]),
            "联合得分": float(metrics["objective"]),
            "是否通过滚动约束": bool(metrics["walk_forward_passed"]),
            "平均验证年化收益": float(metrics["mean_validation_annual_return"]),
            "平均验证夏普": float(metrics["mean_validation_sharpe"]),
            "验证夏普标准差": float(metrics["validation_sharpe_std"]),
            "最差验证最大回撤": float(metrics["worst_validation_drawdown"]),
            "平均验证年化总换手": float(metrics["mean_validation_annual_turnover"]),
            "平均验证年化总换手（百分比）": float(metrics["mean_validation_annual_turnover"] * 100.0),
            "配置JSON": json.dumps(config, ensure_ascii=False, sort_keys=True),
        }
        for cluster_name, config_item in config.items():
            prefix = configs_display_name(cluster_name)
            row[f"{prefix}_资金锚"] = float(config_item["capital_weight"])
            row[f"{prefix}_市值组"] = ",".join(map(str, config_item["market_cap_groups"]))
            row[f"{prefix}_选股比例"] = float(config_item["select_pct"])
            row[f"{prefix}_簇内调仓日"] = int(config_item["rebalance_days"])
            if config_item.get("use_defensive_compensation", False):
                row[f"{prefix}_趋势均线"] = int(config_item["trend_ma_window"])
                row[f"{prefix}_风险关闭因子仓位"] = float(config_item["risk_off_factor_exposure"])
        c1 = config["cluster_1_defensive_small"]["factor_weights"]
        c2 = config["cluster_2_value_growth"]["factor_weights"]
        row.update({
            "防御簇_HML权重": float(c1["hml_r_std_5m"]),
            "防御簇_动量权重": float(c1["exp_wgt_return_6m"]),
            "防御簇_换手权重": float(c1["bias_std_turn_5d"]),
            "价值簇_BP权重": float(c2["BP"]),
            "价值簇_成长权重": float(c2["Profit_G_q"]),
        })
        rows.append(row)
    frame = pd.DataFrame(rows)
    return frame.sort_values(
        ["是否通过滚动约束", "联合得分", "平均验证夏普"],
        ascending=[False, False, False],
    ).reset_index(drop=True) if not frame.empty else frame


def run_joint_periodic_search(cache: Mapping[str, object]):
    """运行联合 TPE，并保存每个 trial 的完整配置、影子账户、验证折指标和触发计划。"""
    if optuna is None:
        raise ImportError("联合滚动优化需要 Optuna，请先在 BigQuant 环境安装 optuna。")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    sampler = optuna.samplers.TPESampler(
        seed=JOINT_OPTUNA_SEED, multivariate=True, n_startup_trials=80,
    )
    study = optuna.create_study(direction="maximize", sampler=sampler)
    records: Dict[int, dict] = {}
    started = time.time()

    def objective(trial):
        configs = suggest_optimized_configs(trial)
        periodic_days = int(trial.suggest_int(
            "periodic_rebalance_days", *JOINT_PERIODIC_REBALANCE_RANGE,
            step=JOINT_PERIODIC_REBALANCE_STEP,
        ))
        anchor = static_anchor_weights(configs)
        shadow_returns, shadow_turnovers = build_joint_cluster_shadow_inputs(configs, cache)
        simulation, decisions = simulate_joint_periodic_portfolio(
            shadow_returns, shadow_turnovers, anchor, periodic_days
        )
        metrics, folds = calculate_joint_walk_forward_metrics(simulation)
        records[trial.number] = {
            "config": deepcopy(configs), "periodic_days": periodic_days,
            "anchor": anchor.copy(), "shadow_returns": shadow_returns,
            "shadow_turnovers": shadow_turnovers, "simulation": simulation,
            "decisions": decisions, "metrics": metrics, "folds": folds,
        }
        for key, value in metrics.items():
            if isinstance(value, (int, float, bool, np.integer, np.floating)):
                trial.set_user_attr(key, float(value))
        return float(metrics["objective"])

    def callback(study_object, frozen_trial):
        completed = sum(item.value is not None for item in study_object.trials)
        total = int(JOINT_OPTUNA_N_TRIALS)
        if completed == 1 or completed == total or completed % 10 == 0:
            elapsed = time.time() - started
            eta = elapsed / completed * (total - completed) if completed else np.nan
            best = records[study_object.best_trial.number]
            progress(
                f"[联合TPE] {completed}/{total}（{completed / total:.1%}），"
                f"已用{elapsed / 60:.1f}分钟，预计剩余{eta / 60:.1f}分钟；"
                f"当前最优：周期={best['periodic_days']}日、"
                f"平均夏普={best['metrics']['mean_validation_sharpe']:.3f}、"
                f"最差回撤={best['metrics']['worst_validation_drawdown']:.2%}"
            )

    study.optimize(objective, n_trials=int(JOINT_OPTUNA_N_TRIALS), callbacks=[callback], gc_after_trial=True)
    if not records:
        raise RuntimeError("联合滚动优化没有得到有效试验。")
    results = joint_periodic_results_frame(study, records)
    passed = results[results["是否通过滚动约束"].astype(bool)]
    if passed.empty:
        raise RuntimeError("所有联合候选均未通过回撤或全股票换手约束，请扩大搜索空间或调整约束。")
    best_trial = int(passed.iloc[0]["试验编号"])
    best = deepcopy(records[best_trial])
    best["trial_number"] = best_trial
    fold_results = pd.concat(
        [record["folds"].assign(试验编号=number, 周期再平衡间隔=record["periodic_days"])
         for number, record in records.items()], ignore_index=True,
    )
    return study, results, fold_results, best


def joint_periodic_sensitivity(best: Mapping[str, object]) -> pd.DataFrame:
    """最优配置固定，仅扰动周期；该报告不参与重新选参。"""
    base_period = int(best["periodic_days"])
    candidates = sorted({
        int(np.clip(base_period + delta, *JOINT_PERIODIC_REBALANCE_RANGE))
        for delta in (-15, -10, -5, 0, 5, 10, 15)
    })
    rows = []
    for period in candidates:
        simulation, decisions = simulate_joint_periodic_portfolio(
            best["shadow_returns"], best["shadow_turnovers"], best["anchor"], period
        )
        metrics, _ = calculate_joint_walk_forward_metrics(simulation)
        rows.append({
            "周期再平衡间隔": period,
            "是否为最优周期": bool(period == base_period),
            "验证期触发次数": int(len(decisions[
                (pd.to_datetime(decisions["决策日期"]) >= pd.Timestamp("2022-01-01"))
                & (pd.to_datetime(decisions["决策日期"]) <= pd.Timestamp("2024-12-31"))
            ])) if not decisions.empty else 0,
            "联合得分": float(metrics["objective"]),
            "平均验证年化收益": float(metrics["mean_validation_annual_return"]),
            "平均验证夏普": float(metrics["mean_validation_sharpe"]),
            "最差验证最大回撤": float(metrics["worst_validation_drawdown"]),
            "平均验证年化总换手": float(metrics["mean_validation_annual_turnover"]),
            "平均验证年化总换手（百分比）": float(metrics["mean_validation_annual_turnover"] * 100.0),
        })
    return pd.DataFrame(rows)


def plot_joint_periodic_search(results: pd.DataFrame, sensitivity: pd.DataFrame) -> str:
    configure_chinese_font()
    figure, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    ordered = results.sort_values("试验编号")
    axes[0].plot(ordered["试验编号"], ordered["联合得分"], color="#9DC3E6", alpha=0.65, label="单次得分")
    axes[0].plot(ordered["试验编号"], ordered["联合得分"].cummax(), color="#C44E52", linewidth=2, label="历史最优")
    axes[0].set_title("联合滚动优化搜索过程")
    axes[0].set_xlabel("试验编号")
    axes[0].set_ylabel("联合得分")
    axes[0].grid(alpha=0.25)
    axes[0].legend()
    axes[1].plot(sensitivity["周期再平衡间隔"], sensitivity["平均验证夏普"], marker="o", label="平均夏普")
    axes[1].plot(sensitivity["周期再平衡间隔"], sensitivity["联合得分"], marker="s", label="联合得分")
    axes[1].set_title("最优配置的周期敏感性（不参与选参）")
    axes[1].set_xlabel("周期再平衡间隔（交易日）")
    axes[1].grid(alpha=0.25)
    axes[1].legend()
    figure.tight_layout()
    path = os.path.join(result_output_directory(), "联合滚动优化搜索与敏感性图.png")
    figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    return path


def run_joint_periodic_walk_forward_optimized_strategy():
    """默认研究入口：缓存数据 → 联合 TPE → 固定最佳参数 → 一次原生确认。

    2022--2024 验证折只用于选参。最优参数确定后，原生回测覆盖 START_DATE 至 END_DATE，
    因而包含 2025--2026 冻结确认期；确认期结果不会反向改变任何参数或评分规则。
    """
    global _ACTIVE_CONFIGS
    configure_chinese_font()
    progress("[联合优化 0/7] 创建中文结果目录：" + result_output_directory())
    _ACTIVE_CONFIGS = validate_configs(CLUSTER_CONFIGS)
    progress("[联合优化 1/7] 一次性准备因子、排名、行情和交易约束缓存")
    cache = prepare_optimization_cache(_ACTIVE_CONFIGS)
    progress("[联合优化 2/7] 对四簇参数、资金锚和周期再平衡间隔执行滚动TPE搜索")
    study, results, fold_results, best = run_joint_periodic_search(cache)
    sensitivity = joint_periodic_sensitivity(best)
    best_row = results[results["试验编号"] == int(best["trial_number"])].head(1).copy()
    if best_row.empty:
        raise RuntimeError("最优 trial 未能在联合试验结果中定位，已停止以避免保存不完整交接文件。")
    paths = {
        "全部联合试验": save_result_table(results, "联合滚动优化全部试验.csv"),
        "各折验证明细": save_result_table(fold_results, "联合滚动优化各折验证明细.csv"),
        "最优配置": save_result_json({
            "周期再平衡间隔": int(best["periodic_days"]),
            "静态资金锚": {name: float(value) for name, value in zip(best["config"], best["anchor"])},
            "四簇配置": best["config"],
        }, "联合滚动优化最优配置.json"),
        "最优周期敏感性": save_result_table(sensitivity, "联合滚动优化最优周期敏感性.csv"),
        "最优周期触发计划": save_result_table(best["decisions"], "联合滚动优化最优周期触发计划.csv"),
        "搜索与敏感性图": plot_joint_periodic_search(results, sensitivity),
        "最优参数摘要": save_result_table(best_row, "联合滚动优化最优参数摘要.csv"),
    }
    display(best_row.round(6))

    progress("[联合优化 3/7] 用最佳联合配置生成原生信号与成交级簇账本")
    _ACTIVE_CONFIGS = validate_configs(best["config"])
    prepare_strategy_data_from_optimization_cache(_ACTIVE_CONFIGS, cache)
    _RUNTIME_DATA["initial_cluster_capital_weights"] = {
        name: float(value) for name, value in zip(_ACTIVE_CONFIGS, best["anchor"])
    }
    _RUNTIME_DATA["band_rebalance_anchor_weights"] = best["anchor"].copy()
    _RUNTIME_DATA["band_rebalance_params"] = {"周期再平衡间隔": int(best["periodic_days"])}
    RESEARCH_ARTIFACTS.update({
        "joint_periodic_study": study,
        "joint_periodic_results": results,
        "joint_periodic_fold_results": fold_results,
        "joint_periodic_best": deepcopy(best),
        "joint_periodic_output_paths": paths,
        "band_rebalance_runtime_records": [],
        "band_rebalance_state_records": [],
    })
    for key in ("rank_cache", "market_matrices", "base_panel", "factor_panels", "candidate_instruments", "trend_close"):
        cache.pop(key, None)
    gc.collect()

    progress("[联合优化 4/7] 运行最优候选的一次 BigTrader 原生确认回测")
    performance = run_prepared_bigtrader_band_rebalance()
    summary = performance_summary_frame(performance)
    runtime_decisions = pd.DataFrame(RESEARCH_ARTIFACTS.get("band_rebalance_runtime_records", []))
    runtime_states = pd.DataFrame(RESEARCH_ARTIFACTS.get("band_rebalance_state_records", []))
    consistency = band_trigger_consistency_audit(best["decisions"], runtime_decisions)
    ledger_daily = pd.DataFrame(RESEARCH_ARTIFACTS.get("cluster_ledger_daily_records", []))
    paths.update({
        "最终原生回测摘要": save_result_table(summary, "最终原生回测摘要.csv"),
        "原生周期触发明细": save_result_table(runtime_decisions, "原生周期再平衡触发明细.csv"),
        "原生日度状态审计": save_result_table(runtime_states, "原生周期再平衡日度状态审计.csv"),
        "快速与原生触发一致性": save_result_table(consistency, "快速与原生触发一致性审计.csv"),
    })
    position_chart = plot_band_native_account_positions(ledger_daily)
    if position_chart:
        paths["四簇账户仓位图"] = position_chart
    RESEARCH_ARTIFACTS["joint_periodic_output_paths"] = paths
    RESEARCH_ARTIFACTS["final_performance_summary"] = summary
    progress("[联合优化 7/7] 完成。结果目录：" + result_output_directory())
    return {
        "study": study, "best_config": best["config"],
        "periodic_days": int(best["periodic_days"]), "performance": performance,
        "output_directory": result_output_directory(),
    }


def run_fixed_best_parameter_strategy():
    """固定 trial 285 的参数运行原生 BigTrader，不调用任何 TPE、影子账户或敏感性分析。"""
    global _ACTIVE_CONFIGS
    _ACTIVE_CONFIGS = validate_configs(CLUSTER_CONFIGS)
    progress(
        "[固定策略 1/3] 准备最优参数的点时因子、股票池、调仓日程和趋势状态；"
        f"运行区间：{START_DATE} 至 {END_DATE}"
    )
    prepare_strategy_data(_ACTIVE_CONFIGS)

    # 此处的资金锚和周期均来自联合 TPE 最优 trial 285；运行中绝不根据绩效动态改配。
    anchor = static_anchor_weights(_ACTIVE_CONFIGS)
    _RUNTIME_DATA["initial_cluster_capital_weights"] = {
        name: float(value) for name, value in zip(_ACTIVE_CONFIGS, anchor)
    }
    _RUNTIME_DATA["band_rebalance_anchor_weights"] = anchor.copy()
    _RUNTIME_DATA["band_rebalance_params"] = {
        "周期再平衡间隔": int(FIXED_PERIODIC_REBALANCE_DAYS)
    }
    RESEARCH_ARTIFACTS.update({
        "band_rebalance_runtime_records": [],
        "band_rebalance_state_records": [],
    })

    progress("[固定策略 2/3] 启动 BigTrader：T 日收盘形成信号，T+1 日开盘执行")
    performance = bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date=START_DATE,
        end_date=END_DATE,
        capital_base=CAPITAL_BASE,
        benchmark=BENCHMARK,
        initialize=initialize_band_rebalance,
        handle_data=handle_data_band_rebalance,
        after_trading=after_trading_band_rebalance,
        handle_order=handle_order,
        handle_trade=handle_trade,
        order_price_field_buy="open",
        order_price_field_sell="open",
        volume_limit=VOLUME_LIMIT,
    )
    progress("[固定策略 3/3] BigTrader 运行完成")
    try:
        display(performance.summary)
    except Exception:
        display(performance)
    return performance


if __name__ == "__main__":
    performance = run_fixed_best_parameter_strategy()


[加载 1/4] 已进入策略脚本，正在加载科学计算组件
[加载 2/4] 科学计算组件加载完成，正在检查 Optuna
[加载 3/4] Optuna 检查完成，正在连接 BigQuant 组件
[加载 4/4] BigQuant 组件加载完成，正在初始化策略参数
[00:00] [固定策略 1/3] 准备最优参数的点时因子、股票池、调仓日程和趋势状态；运行区间：2020-01-01 至 2026-06-30
[00:00] [1/7] 获取交易日历并建立四套独立调仓日程
[00:00] 调仓截面数量：防御型小市值综合因子=63；价值成长综合因子=79；大市值财务质量=79；快速资金流向=524
[00:00] [2/7] 查询所有策略簇共用的点时股票池截面
[00:10] 股票池基础字段行业来源：sw2021_level1
[00:13] 因子预热：2017-08-10至2020-01-01，共使用584个回测开始日前交易日（最长窗口504日）
[00:13] [3/7] 按原始代码公式从基础字段构建七个因子（不查询同名成品因子）
[00:21] hml_r_std_5m构建来源：SQL显式历史区间/pre_close
[00:27] exp_wgt_return_6m构建来源：cn_stock_prefactors.close/turn + m_lag
[00:36] bias_std_turn_5d构建来源：cn_stock_prefactors.turn / DAI m_stddev(5,504)
[00:37] BP构建来源：cn_stock_valuation.pb；公式=1/PB
[00:39] Profit_G_q数据来源：cn_stock_prefactors.net_profit_yoy_mrq（净利润同比增长率（单季度））
[00:39] qfa_roe构建来源：cn_stock_prefactors.roe_avg_mrq（单季度平均ROE原始字段）
[00:45] mfd_sellamt_d(10日累计)构建来源：cn_stock_moneyflow.outflow_amount_main / SQL滚动10日求和；公式=-rolling_sum(10)
[00:46] 因子覆盖：BP，2020-01-02至2026-06-15，79个截面

[2026-07-24 02:34:59] [info     ] bigtrader run done.
[04:19] [固定策略 3/3] BigTrader 运行完成


{'return_ratio': 301.26,
 'annual_return_ratio': 24.97,
 'benchmark_ratio': 19.92,
 'beta': 0.51,
 'alpha': 0.22,
 'sharp_ratio': 1.2,
 'ir': 1.11,
 'return_volatility': 17.33,
 'max_drawdown': 13.97,
 'win_ratio': 54.31,
 'profit_loss_ratio': 2.0}